In [ ]:
%reset -f
import math
import os
import warnings
from collections.abc import Callable
from math import pow, sqrt

import forallpeople as units
import handcalcs
import handcalcs.render
import numpy as np

# import polars as pl
import pandas as pd
import sympy as sp
from structural_tools.notebook.display import (
    display_table,
    display_text,
)
from structural_tools.notebook.calculation import (
    check_value,
    set_params_columns,
)
    check_value,  # noqa: F401
    display_table,
    display_text,
    set_params_columns,
    sig_figs,  # noqa: F401
)
from structural_tools.notebook import feet_inches as fi
from structural_tools.seismic.asce.parameters import LOAD_FACTOR_SEISMIC_ASD, RiskCategory, SeismicParameters, SiteClass

units.environment(env_name="structural", top_level=True)
handcalcs.set_option("custom_symbols", {"star": "^*", "__": ",", "plus": "+", "minus": "-"})
handcalcs.set_option("greek_exclusions", ["psi"])
handcalcs.set_option("display_precision", 4)
handcalcs.set_option("param_columns", 1)
handcalcs.set_option("latex_block_start", "\\begin{equation*}")
handcalcs.set_option("latex_block_end", "\\end{equation*}")

# Effective Seismic Weight Calculation

We calculate the effective seismic weight with unit weights and square footages.

## Unit Weights

Weights will be assigned with area unit weights.

In [2]:
set_params_columns(4)

In [3]:
%%render params
U_floor = 35 / 1000 * ksf
U_pavers = 15 / 1000 * ksf
U_roof = 25 / 1000 * ksf
U_wall = 12 / 1000 * ksf

<IPython.core.display.Latex object>

## Building Geometry

The lengths and widths of the weight areas are consistent across levels 2, 3, and 4.

In [4]:
set_params_columns(5)

In [5]:
# Plan dimensions
l_C1__C2 = fi(31, 4)
l_C2__C3 = fi(31, 5)
l_C3__C4 = fi(26, 5)
l_C4__C5 = fi(17, 11)
#
l_C6__C7 = fi(21, 4)
l_C7__C8 = fi(26, 6)
l_C8__C9 = fi(36, 5, 1 / 2)

l_C1__C21 = fi(35, 1, 1 / 2)
l_C21__C29 = fi(24, 8)
l_C29__C42 = fi(34, 4)
l_C42__C55 = fi(24, 8)
#
l_C59__C7 = fi(21, 4)
l_C7__C83 = fi(36, 10)
l_C83__C9 = fi(26, 1, 1 / 2)

l_C3__C33 = fi(8, 10)
l_C33__C4 = l_C3__C4 - l_C3__C33
l_C7__C73 = fi(7, 7, 1 / 4)
l_C6__C73 = l_C6__C7 + l_C7__C73
l_C73__C8 = l_C7__C8 - l_C7__C73
l_C8__C81 = fi(1, 5, 0.75)
l_C81__C83 = fi(8, 10)
l_C7__C81 = l_C7__C8 + l_C8__C81

l_CA__CB = fi(30, 9)
l_CB__CC = fi(5, 6)
l_CC__CD = fi(30, 9)

l_CW__CX = fi(30, 9)
l_CX__CY = fi(5, 6)
l_CY__CZ = fi(30, 9)

l_CW__CW6 = fi(19, 11, 0.25)
l_CW6__CX = l_CW__CX - l_CW__CW6
l_CY__CY6 = fi(19, 8, 0.25)
l_CY6__CZ = l_CY__CZ - l_CY__CY6

The wall heights vary across the levels.

In [6]:
%%render params
h_wall__1 = 13.5 * ft
h_wall__2 = 10.5 * ft
h_wall__3 = 10.5 * ft
h_wall__4 = 10.5 * ft
h_wall__5 = 10.5 * ft

<IPython.core.display.Latex object>

In [7]:
P_wall = (
    7
    + fi(30, 9)
    + fi(79, 9, 1 / 2)
    + fi(86, 1, 1 / 4)
    + fi(30, 9)
    + fi(31, 4)
    + fi(36, 3)
    + fi(131, 3 / 4)
    + fi(100, 4, 3 / 4)
    + fi(36, 3)
) * ft

In [8]:
%%render
P_wall

<IPython.core.display.Latex object>

The areas of the building are:

In [9]:
%%render params
A_level = 13417 * ft**2
A_pavers = 907 * ft**2
A_floor = A_level - A_pavers

<IPython.core.display.Latex object>

In [10]:
%%render
A_wall__1 = h_wall__1 * P_wall
A_wall__2 = h_wall__2 * P_wall
A_wall__3 = h_wall__3 * P_wall
A_wall__4 = h_wall__4 * P_wall
A_wall__5 = h_wall__5 * P_wall

<IPython.core.display.Latex object>

## Component Weights

The typical floor and corridor weights are

In [11]:
%%render
W_topfloor = U_floor * A_floor
W_pavers = (U_floor + U_pavers) * A_pavers
W_floor__typ = U_floor * A_level

<IPython.core.display.Latex object>

This applies to levels 2, 3, and 4. The 5th (top) level has pavers in the N-E corner.

In [12]:
set_params_columns(4)

In [13]:
%%render params
W_floor__2 = W_floor__typ
W_floor__3 = W_floor__typ
W_floor__4 = W_floor__typ
W_floor__5 = W_topfloor + W_pavers

<IPython.core.display.Latex object>

The roof weight is

In [14]:
%%render
W_roof = U_roof * A_floor

<IPython.core.display.Latex object>

The exterior walls are divided in half to the floor above and below.

In [15]:
%%render
W_wall__2 = U_wall * (A_wall__2 / 2 + A_wall__1 / 2)
W_wall__3 = U_wall * (A_wall__3 / 2 + A_wall__2 / 2)
W_wall__4 = U_wall * (A_wall__4 / 2 + A_wall__3 / 2)
W_wall__5 = U_wall * (A_wall__5 / 2 + A_wall__4 / 2)
W_wall__roof = U_wall * (A_wall__5 / 2)

<IPython.core.display.Latex object>

## Level Weights

Each level's weight is

In [16]:
%%render 
W_level__2 = W_floor__2 + W_wall__2
W_level__3 = W_floor__3 + W_wall__3
W_level__4 = W_floor__4 + W_wall__4
W_level__5 = W_floor__5 + W_wall__5
W_level__roof = W_roof + W_wall__roof

<IPython.core.display.Latex object>

## Total Weight

The total weight of the structure is

In [17]:
%%render long
W_total = W_level__2 + W_level__3 + W_level__4 + W_level__5 + W_level__roof

<IPython.core.display.Latex object>

# Seismic Equivalent Lateral Force (ELF) Analysis

We will be using the equivalent lateral force method prescribed by ASCE 7-16 Section 12.8. We will envelope the forces with a fully flexible diaphragm assumption and a fully rigid diaphragm assumption.

## Seismic Parameters

In [18]:
S_S = 0.887  # g
S_1 = 0.410  # g
seismic_params = SeismicParameters(s_1=S_1, s_s=S_S, risk_category=RiskCategory.II, site_class=SiteClass.D)

Seismic design parameters are obtained from the geotech report and ASCE 7-16 Sec. 11 and 12.

In [19]:
set_params_columns(5)

In [20]:
%%render params
R = 6.5
Omega_0 = 3.0
C_d = 4.0
rho = 1.0

<IPython.core.display.Latex object>

In [21]:
%%render params
I_e = seismic_params.i_e
S_S = seismic_params.s_s  # g
S_1 = seismic_params.s_1  # g
F_a = seismic_params.f_a
F_v = seismic_params.f_v
S_MS = seismic_params.s_ms  # g
S_M1 = seismic_params.s_m1  # g
S_DS = seismic_params.s_ds  # g
S_D1 = seismic_params.s_d1  # g
SDC = seismic_params.sdc
C_u = 1.4
T_L = 16  # s

<IPython.core.display.Latex object>

## Base Shear

We must calculate the period

In [22]:
set_params_columns(1)

In [23]:
%%render params
h_top__roof = fi(71, 3) * ft
h_bottom__roof = fi(58, 7) * ft
h_n = np.mean([h_top__roof, h_bottom__roof])  # ASCE 7-16 Sec. 11.2

<IPython.core.display.Latex object>

In [24]:
%%render
h_n__check = check_value(h_n, 65 * ft, "<=")

<IPython.core.display.Latex object>

We will approximate the building period.

In [25]:
h_n = float(h_n)  # to prevent weirdness with units

In [26]:
%%render
C_t = 0.02  # ASCE 7-16 Table 12.8-2
x = 0.75  # ASCE 7-16 Table 12.8-2
T_a = C_t * h_n**x  # ASCE 7-16 Eq. 12.8-7

<IPython.core.display.Latex object>

Because we are not using an automated analysis software, we can make the following assumption:

In [27]:
%%render
T = T_a  # ASCE 7-16 Sec. 12.8.2

<IPython.core.display.Latex object>

With the period, we can calculate the seismic response coefficient.

In [28]:
%%render
C_s = S_DS / (R / I_e)  # ASCE 7-16 Eq. 12.8-2

<IPython.core.display.Latex object>

We must check that $C_s$ does not exceed:

In [29]:
%%render long
# fmt: off
if T <= T_L:  C_s__check__max = S_D1 / (T * (R / I_e));              C_s__check__max = check_value(C_s, C_s__check__max, "leq")  # ASCE 7-16 Sec. 12.8.1.1
elif T > T_L: C_s__check__max = (S_D1 * T_L) / (T**2 * (R / I_e));   C_s__check__max = check_value(C_s, C_s__check__max, "leq")  # ASCE 7-16 Sec. 12.8.1.1
# fmt: on

<IPython.core.display.Latex object>

And we must also check that $C_s$ is not less than:

In [30]:
%%render
C_s__check__min__1 = 0.044 * S_DS * I_e  # ASCE 7-16 Eq. 12.8-5
C_s__check__min = check_value(C_s, C_s__check__min__1, "geq")

<IPython.core.display.Latex object>

as well as:

In [31]:
%%render
C_s__check__min = check_value(C_s, 0.01, "geq")  # ASCE 7-16 Eq. 12.8-5

<IPython.core.display.Latex object>

and finally:

In [32]:
%%render
# fmt: off
if S_1 >= 0.6:  C_s__check__min__3 = 0.5 * S_1 / (R / I_e); C_s__check__min = check_value(C_s, C_s__check__min__3, "geq")  # ASCE 7-16 Eq. 12.8-6
elif S_1 < 0.6: C_s__check__min = "\\text{skipping check}"  # ASCE 7-16 Eq. 12.8-6
# fmt: on

<IPython.core.display.Latex object>

And now, we can calculate the base shear, $V$

In [33]:
%%render
V = C_s * W_total  # ASCE 7-16 Eq. 12.8-1

<IPython.core.display.Latex object>

## Distribute the seismic forces

With the base shear calculated, we must now distribute the base shear on each level.

First, we must calculate the structural period exponent, $k$.

In [34]:
%%render
# fmt: off
if T <= 0.5: k = 1
elif T >= 2.5: k = 2
elif T > 0.5 and T < 2.5: k = 1 + ((T - 0.5) * (2 - 1)) / (2.5 - 0.5)
# fmt: on

<IPython.core.display.Latex object>

Now, we can distribute the forces per floor,

$$
F_x = C_{vx} V\;\;\text{(ASCE 7-16 Eq. 12.8-11)}
$$

As well as cumulatively from roof to ground.

$$
V_x = \sum_{i=x}^n F_i\;\; \text{(ASCE 7-16 Eq. 12.8-11)}
$$

In [ ]:
# Load in basic data
seismic_loads = pd.DataFrame(
    {
        "Level": ["Level 1", "Level 2", "Level 3", "Level 4", "Level 5", "Roof"],
        "floor height": [
            float(h_wall__1),
            float(h_wall__2),
            float(h_wall__3),
            float(h_wall__4),
            float(h_wall__5),
            0,
        ],
        "floor weight": [
            0,
            float(W_level__2),
            float(W_level__3),
            float(W_level__4),
            float(W_level__5),
            float(W_level__roof),
        ],
    },
)
seismic_loads = seismic_loads.set_index("Level")

# Calc cumulative elevation
elevations = [0]
for height in seismic_loads["floor height"][:-1]:
    elevations.append(height + elevations[-1])
seismic_loads.insert(2, "elevation", elevations)

# Calc coefficients and story forces ASCE 7-16 Eq. 12.8-11 and 12.8-12
seismic_loads["wxhxk"] = seismic_loads["floor weight"] * seismic_loads["elevation"] ** float(k)
cumulative_wxhxk = seismic_loads["wxhxk"].sum()
seismic_loads["C_vx"] = seismic_loads["wxhxk"] / cumulative_wxhxk
seismic_loads["F_x"] = seismic_loads["C_vx"] * float(V)

# Calc story shear forces ASCE 7-16 Eq. 12.8-13
story_shears = [0]
for shear in reversed(list(seismic_loads["F_x"][1:])):
    story_shears.insert(0, shear + story_shears[0])
seismic_loads["V_x"] = story_shears

# Calc overturning moments
floor_moments = seismic_loads["V_x"] * seismic_loads["floor height"]
overturning_moments = [0]
for floor_moment in reversed(list(floor_moments[:-1])):
    overturning_moments.insert(0, overturning_moments[0] + floor_moment)
seismic_loads["OTM"] = overturning_moments

In [36]:
# Rename columns for display with LaTeX formatting for units and subscripts/superscripts
column_name_map = {
    "floor height": "$h_{floor}$ [ft]",
    "floor weight": "$w_x$ [kip]",
    "elevation": "$h_x$ [ft]",
    "wxhxk": "$w_x h_x^k$",
    "C_vx": "$C_{vx}$",
    "F_x": "$F_x$ [kip]",
    "V_x": "$V_x$ [kip]",
    "OTM": "OTM [kip-ft]",
}

Calculating for each floor leads to the following \autoref{tab:seismic_forces_calculation_table}

In [37]:
# Format the dataframe
# Reverse the order of the dataframe with '.loc[::-1]' so roof is the top row and ground is the bottom row, then apply formatting with the 'format_dataframe' function
display_table(
    dataframe=seismic_loads.iloc[::-1],
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Seismic Forces Calculation Table",
    label="tab:seismic_forces_calculation_table",
)

,$h_{floor}$ [ft],$w_x$ [kip],$h_x$ [ft],$w_x h_x^k$,$C_{vx}$,$F_x$ [kip],$V_x$ [kip],OTM [kip-ft]
Level,,,,,,,,
Roof,0.0,348.640312,55.5,19349.537344,0.231894,61.317250,0.000000,0.000000
Level 5,10.5,554.980625,45.0,24974.128125,0.299302,79.141161,61.317250,643.831124
Level 4,10.5,541.375625,34.5,18677.459063,0.223840,59.187484,140.458411,2118.644441
Level 3,10.5,541.375625,24.0,12993.015000,0.155715,41.173902,199.645895,4214.926336
Level 2,10.5,551.630000,13.5,7447.005000,0.089249,23.599007,240.819796,6743.534199
Ground,13.5,0.000000,0.0,0.000000,0.000000,0.000000,264.418803,10313.188045


# Diaphragm and Shear Wall Design

## Redundancy Factor

The redundancy factor, $\rho$, is either 1.3 or 1.0 according to ASCE 7-16 12.3.4.

No single story resists more than 35% of the shear force (see $C_{vx}$ in \autoref{tab:seismic_forces_calculation_table}), so all values of $\rho$ are $1.0$.

## Shear Wall Design

We will envelope the forces with a fully flexible diaphragm assumption and a fully rigid diaphragm assumption.

In [ ]:
# Set up diaphragm dataframe
diaphragm_rigidity = ["Flexible", "Rigid"]
levels = ["Level 5", "Level 4", "Level 3", "Level 2", "Level 1"]
walls = {
    # N-S
    "C-2.1": "N-S",
    "C-2.9": "N-S",
    "C-3": "N-S",
    "C-3.3": "N-S",
    "C-4": "N-S",
    "C-4.2": "N-S",
    "C-5": "N-S",
    "C-5.5": "N-S",
    "C-5.9": "N-S",
    "C-6": "N-S",
    "C-7": "N-S",
    "C-7.3": "N-S",
    # "C-8--C-W": "N-S",
    "C-8--C-W.6": "N-S",
    "C-8.1": "N-S",
    "C-8.3": "N-S",
    # E-W
    "C-B--C-2": "E-W",
    "C-B--C-4": "E-W",
    "C-B--C-4.9": "E-W",
    "C-W.6--C-7.3": "E-W",
    # "C-X--C-5.8": "E-W",
    "C-X--C-6.5": "E-W",
    "C-C--C-2.5": "E-W",
    "C-C--C-3.6": "E-W",
    "C-C--C-5.2": "E-W",
    # "C-Y--C-5.8": "E-W",
    "C-Y--C-6.5": "E-W",
    "C-Y--C-7.2": "E-W",
}

index = pd.MultiIndex.from_product(
    [levels, walls.keys()],
    names=["Level", "Wall"],
)

shear_walls = pd.DataFrame(index=index)
shear_walls["Direction"] = shear_walls.index.get_level_values("Wall").map(walls)
flexible = shear_walls.copy()
flexible_ns = flexible[flexible["Direction"] == "N-S"].copy()
flexible_ew = flexible[flexible["Direction"] == "E-W"].copy()
rigid = shear_walls.copy()
rigid_ns = rigid[rigid["Direction"] == "N-S"].copy()
rigid_ew = rigid[rigid["Direction"] == "E-W"].copy()

### Flexible Diaphragm Assumption

We will uniformly distribute the shear forces across the diaphragms based on tributary areas of each level. This removes some nuance in the distribution of floor weight distribution (paver weight vs. floor weight, for instance), but the vast majority of the area of each level is typical floor weight.

Due to this building's geometry, we must include an additional 

$$
f_{p,x} = F_x / A_{level}
$$

In [39]:
%%render
A_level
f_p__roof = seismic_loads.loc["Roof", "F_x"] * 1000 * lb / A_level
f_p__5 = seismic_loads.loc["Level 5", "F_x"] * 1000 * lb / A_level
f_p__4 = seismic_loads.loc["Level 4", "F_x"] * 1000 * lb / A_level
f_p__3 = seismic_loads.loc["Level 3", "F_x"] * 1000 * lb / A_level
f_p__2 = seismic_loads.loc["Level 2", "F_x"] * 1000 * lb / A_level

<IPython.core.display.Latex object>

#### North-South Direction

Most of the shear walls are not in line which presents an issue: if we assume a diaphragm is the full depth, there will be an extremely long collector element without a shear wall on both sides. We will avoid this by just assuming the flexible diaphragm does not extend the full depth and instead is just tributary to the nearest wall (\autoref{fig:diaphragm_trib_area}). This means that there are essentially two diaphragms through the depth of the building. This is a rough approximation to ensure all the shear load is resolved in the shear walls. \

Proper detailing will be done to ensure continuity through the depth of the diaphragm.

![Shear walls (thick red lines) in plan view with their (colored) tributary areas.](images/shear_wall_tributary_area.png){width=60% #fig:diaphragm_trib_area}

Shear wall shear forces are tabulated in \autoref{tab:shear_wall_forces_roof_ns} through \autoref{tab:shear_wall_forces_level2_ns}.

Shear wall sheathing will be assumed to have the following properties:

1. WSP Sheathing
2. 15/32" thickness
3. 10d common nails
4. Plywood panels
5. Framing uses Douglas-Fir-Larch (no specific gravity adjustment)

Shear capacity and stiffness values come from **AWC SDPWS 2021 Table 4.3A**.

Notes: 

- Cumulative shear force, $F_{x,cum}$, is cumulative from the Roof downwards. For example, Level 3 includes the shear forces from the Roof, Level 4, and Level 3.

In [40]:
from structural_tools.seismic.timber.sheathing import (
    DesignMethod,
    LoadCase,
    Nail,
    PanelType,
    Sheathing,
    SheathingMaterial,
    get_sheathing_properties,
)

# Set baseline sheathing properties. We set the number of sheathed sides later
sheathing = Sheathing(
    sheathing_material=SheathingMaterial.WSP_SHEATHING,
    minimum_nominal_panel_thickness=15 / 32,
    nail=Nail.COMMON_10D,
    panel_type=PanelType.PLY,
)

In [41]:
d_trib_ns = 67 / 2 * ft

In [42]:
# Wall tributary widths and lengths

t_w__C21 = l_C1__C21 + (l_C21__C29 / 2)
t_w__C29 = (l_C21__C29 / 2) + (l_C29__C42 / 2)
t_w__C42 = (l_C29__C42 / 2) + (l_C42__C55 / 2)
t_w__C55 = (l_C42__C55 / 2) + 0  # placeholder for angled area that will be manualled added

t_w__C3 = l_C2__C3 + (l_C3__C33 / 2)
t_w__C33 = (l_C3__C33 / 2) + (l_C33__C4 / 2)
t_w__C4 = (l_C33__C4 / 2) + (l_C4__C5 / 2)
t_w__C5 = (l_C4__C5 / 2) + 0  # placeholder for angled area that will be manualled added

t_w__C59 = (l_C59__C7 / 2) + 0  # placeholder for angled area that will be manualled added
t_w__C7 = (l_C59__C7 / 2) + (l_C7__C81 / 2)
t_w__C81 = (l_C7__C81 / 2) + (l_C81__C83 / 2)
t_w__C83 = (l_C81__C83 / 2) + l_C83__C9

t_w__C6 = (l_C6__C73 / 2) + 0  # placeholder for angled area that will be manualled added
t_w__C73 = (l_C6__C73 / 2) + (l_C73__C8 / 2)
t_w__C8 = (l_C73__C8 / 2) + l_C8__C9 - 7  # 7 foot jog

t_l__C21 = float(d_trib_ns)
t_l__C29 = float(d_trib_ns)
t_l__C3 = float(d_trib_ns)
t_l__C33 = float(d_trib_ns)
t_l__C4 = float(d_trib_ns)
t_l__C42 = float(d_trib_ns)
t_l__C5 = float(d_trib_ns)
t_l__C55 = float(d_trib_ns)
t_l__C59 = float(d_trib_ns)
t_l__C6 = float(d_trib_ns)
t_l__C7 = float(d_trib_ns)
t_l__C73 = float(d_trib_ns)
t_l__C8__1 = fi(15, 8)
t_l__C8__2 = fi(17, 10)
t_l__C81 = float(d_trib_ns)
t_l__C83 = float(d_trib_ns)

In [ ]:
flexible_ns.loc["Level 5", "tributary weight"] = float(f_p__roof)
flexible_ns.loc["Level 4", "tributary weight"] = float(f_p__5)
flexible_ns.loc["Level 3", "tributary weight"] = float(f_p__4)
flexible_ns.loc["Level 2", "tributary weight"] = float(f_p__3)
flexible_ns.loc["Level 1", "tributary weight"] = float(f_p__2)

# Nonparallel irregularity
angle_degrees = 23
angle = math.radians(angle_degrees)
angled_walls = ["C-2.1", "C-2.9", "C-3", "C-3.3", "C-4", "C-4.2", "C-5", "C-5.5"]
wall_indices = flexible_ns.index.get_level_values("Wall").isin(angled_walls)
flexible_ns.loc[wall_indices, "tributary weight"] *= math.cos(angle) + 0.3 * math.sin(angle)

# Shear wall height is for the floor below
flexible_ns.loc["Level 5", "wall height"] = float(seismic_loads.loc["Level 5", "floor height"])
flexible_ns.loc["Level 4", "wall height"] = float(seismic_loads.loc["Level 4", "floor height"])
flexible_ns.loc["Level 3", "wall height"] = float(seismic_loads.loc["Level 3", "floor height"])
flexible_ns.loc["Level 2", "wall height"] = float(seismic_loads.loc["Level 2", "floor height"])
flexible_ns.loc["Level 1", "wall height"] = float(seismic_loads.loc["Level 1", "floor height"])

# Add wall tributary widths and wall lengths to df
l_typ = 29
l_stairwall = 19
l_stairwall_short = fi(14, 3)
l_elevator = 11
wall_tributary = {
    "C-2.1": {"trib_width": float(t_w__C21), "trib_depth": t_l__C21, "wall_length": l_typ},
    "C-2.9": {"trib_width": float(t_w__C29), "trib_depth": t_l__C29, "wall_length": l_typ},
    "C-3": {"trib_width": float(t_w__C3), "trib_depth": t_l__C3, "wall_length": l_stairwall_short},
    "C-3.3": {"trib_width": float(t_w__C33), "trib_depth": t_l__C33, "wall_length": l_stairwall},
    "C-4": {"trib_width": float(t_w__C4), "trib_depth": t_l__C4, "wall_length": l_typ},
    "C-4.2": {"trib_width": float(t_w__C42), "trib_depth": t_l__C42, "wall_length": l_typ},
    "C-5": {"trib_width": float(t_w__C5), "trib_depth": t_l__C5, "wall_length": l_typ},
    "C-5.5": {"trib_width": float(t_w__C55), "trib_depth": t_l__C55, "wall_length": l_typ},
    "C-5.9": {"trib_width": float(t_w__C59), "trib_depth": t_l__C59, "wall_length": l_typ},
    "C-6": {"trib_width": float(t_w__C6), "trib_depth": t_l__C6, "wall_length": l_typ},
    "C-7": {"trib_width": float(t_w__C7), "trib_depth": t_l__C7, "wall_length": l_typ},
    "C-7.3": {"trib_width": float(t_w__C73), "trib_depth": t_l__C73, "wall_length": l_elevator},
    # "C-8--C-W": {"trib_width": float(t_w__C8), "trib_depth": t_l__C8__1, "wall_length": fi(9, 6)},
    "C-8--C-W.6": {"trib_width": float(t_w__C8), "trib_depth": t_l__C8__2 + t_l__C8__1, "wall_length": l_elevator},
    "C-8.1": {"trib_width": float(t_w__C81), "trib_depth": t_l__C81, "wall_length": l_stairwall_short},
    "C-8.3": {"trib_width": float(t_w__C83), "trib_depth": t_l__C83, "wall_length": l_stairwall},
}

flexible_ns["wall length"] = flexible_ns.index.get_level_values("Wall").map(
    {key: value["wall_length"] for key, value in wall_tributary.items()}
)
flexible_ns["tributary width"] = flexible_ns.index.get_level_values("Wall").map(
    {key: value["trib_width"] for key, value in wall_tributary.items()}
)
flexible_ns["tributary depth"] = flexible_ns.index.get_level_values("Wall").map(
    {key: value["trib_depth"] for key, value in wall_tributary.items()}
)
flexible_ns["tributary area"] = flexible_ns["tributary depth"] * flexible_ns["tributary width"]

# Fix trib area of certain walls on all levels
flexible_ns.loc[flexible_ns.index.get_level_values("Wall") == "C-2.1", "tributary area"] += 116
flexible_ns.loc[flexible_ns.index.get_level_values("Wall") == "C-5", "tributary area"] += 447
flexible_ns.loc[flexible_ns.index.get_level_values("Wall") == "C-5.5", "tributary area"] += 290
flexible_ns.loc[flexible_ns.index.get_level_values("Wall") == "C-5.9", "tributary area"] += 416
flexible_ns.loc[flexible_ns.index.get_level_values("Wall") == "C-6", "tributary area"] += 195
flexible_ns.loc[flexible_ns.index.get_level_values("Wall") == "C-8.3", "tributary area"] += 23.5

# Ensure that the calculated areas from earlier and tributary areas match
trib_area = flexible_ns["tributary area"].sum() / len(flexible_ns.index.get_level_values("Level").unique())
assert np.isclose(trib_area, float(A_level)), (
    f"Shear wall tributary areas ({trib_area}) and level area ({float(A_level)}) do not match up. Ensure you are including all areas in your calculations"
)

# Automatically extract the shear line by splitting at "_" and taking the first section
flexible_ns["line"] = flexible_ns.index.get_level_values("Wall").str.split("--").str[0]
# Get tributary area and wall length for the shear line
flexible_ns["line tributary area"] = flexible_ns.groupby(["Level", "line"])["tributary area"].transform("sum")
flexible_ns["line wall length"] = flexible_ns.groupby(["Level", "line"])["wall length"].transform("sum")

flexible_ns["line shear demand"] = flexible_ns["line tributary area"] * flexible_ns["tributary weight"]
flexible_ns["line unit shear"] = flexible_ns["line shear demand"] / flexible_ns["line wall length"]
flexible_ns["line cumulative shear demand"] = flexible_ns.groupby(level="Wall")["line shear demand"].cumsum()
flexible_ns["line cumulative unit shear"] = (
    flexible_ns["line cumulative shear demand"] / flexible_ns["line wall length"]
)

flexible_ns["shear demand"] = flexible_ns["line unit shear"] * flexible_ns["wall length"]
flexible_ns["cumulative shear demand"] = flexible_ns.groupby(level="Wall")["shear demand"].cumsum()
flexible_ns["unit shear demand"] = flexible_ns["cumulative shear demand"] / flexible_ns["wall length"]
flexible_ns["adjusted unit shear demand"] = flexible_ns["unit shear demand"] * LOAD_FACTOR_SEISMIC_ASD

flexible_ns["sheathed sides"] = 1
flexible_ns["aspect ratio"] = flexible_ns["wall height"] / flexible_ns["wall length"]

#### Check aspect ratios

We must check the aspect ratios of the shear walls to ensure we don't need to apply the aspect ratio factor in SDPWS Sec. 4.3.3.2. The maximum aspect ratio of all N-S walls is:

In [44]:
%%render
ratio_max = check_value(flexible_ns["aspect ratio"].max(), 2, "<")

<IPython.core.display.Latex object>

In [45]:
if "OK" in ratio_max:
    display_text(r"$\therefore$ All aspect ratios are OK")
else:
    display_text("The aspect ratio knockdown factor must be applied.")

'$\\therefore$ All aspect ratios are OK'

In [46]:
flexible_ns[["adjusted unit shear capacity", "nail spacing", "shear stiffness", "shear dcr"]] = (
    get_sheathing_properties(flexible_ns[["adjusted unit shear demand", "sheathed sides"]], sheathing=sheathing)
)

shear_dcr_check_value = 1

# Assign 2-sided sheathing to walls requiring it, then recalc capacity for just those walls (more efficient than passing the full wall dataframe)
if (flexible_ns["shear dcr"] > shear_dcr_check_value).any():
    flexible_ns.loc[flexible_ns["shear dcr"] > shear_dcr_check_value, "sheathed sides"] = 2
    flexible_ns.loc[
        flexible_ns["shear dcr"] > shear_dcr_check_value,
        ["adjusted unit shear capacity", "nail spacing", "shear stiffness", "shear dcr"],
    ] = get_sheathing_properties(
        flexible_ns.loc[
            flexible_ns["shear dcr"] > shear_dcr_check_value, ["adjusted unit shear demand", "sheathed sides"]
        ],
        sheathing=sheathing,
    )

In [47]:
# Update dataframe
flexible = flexible.reindex(columns=flexible_ns.columns)
flexible["line"] = flexible["line"].astype("string")
flexible.update(flexible_ns)

In [48]:
# Rename columns for display with LaTeX formatting for units and subscripts/superscripts
column_name_map = {
    "tributary weight": "$w_{trib}$ [psf]",
    "tributary area": "$A$ [ft$^2$]",
    "shear demand": "$F_x$ [lbf]",
    "cumulative shear demand": "$F_{x,cum}$ [lbf]",
    "wall length": "$l_{wall}$ [ft]",
    "unit shear demand": "$v$ [plf]",
    "adjusted unit shear demand": "$0.7 v$ [lbf]",
    "adjusted unit shear capacity": "$v_{cap}$ [plf]",
    "sheathed sides": "sides",
    "nail spacing": "$s_{nail}$ [in]",
}

In [49]:
def highlight_insufficient_capacity(row):
    styles = [""] * len(row)

    if row["$0.7 v$ [plf]"] > row["$v_{cap}$ [plf]"]:
        demand_idx = row.index.get_loc("$0.7 v$ [plf]")
        capacity_idx = row.index.get_loc("$v_{cap}$ [plf]")

        styles[demand_idx] = "background-color: red; font-weight: bold"
        styles[capacity_idx] = "background-color: red; font-weight: bold"

    return styles

In [ ]:
display_table(
    dataframe=flexible_ns,
    levels="Level 5",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Shear wall forces NS - Level 5",
    label="tab:shear_wall_forces_roof_ns",
)
display_table(
    dataframe=flexible_ns,
    levels="Level 4",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Shear wall forces NS - Level 4",
    label="tab:shear_wall_forces_level4_ns",
)
display_table(
    dataframe=flexible_ns,
    levels="Level 3",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Shear wall forces NS - Level 3",
    label="tab:shear_wall_forces_level3_ns",
)
display_table(
    dataframe=flexible_ns,
    levels="Level 2",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Shear wall forces NS - Level 2",
    label="tab:shear_wall_forces_level2_ns",
)
display_table(
    dataframe=flexible_ns,
    levels="Level 1",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Shear wall forces NS - Level 1",
    label="tab:shear_wall_forces_level1_ns",
)

,$w_{trib}$ [psf],$A$ [ft$^2$],$F_x$ [lbf],"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [lbf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,,,,
C-2.1,4.742520,1705.854167,8090.047854,8090.047854,29.00,278.967167,195.277017,310.714286,1,6.0
C-2.9,4.742520,988.250000,4686.795594,4686.795594,29.00,161.613641,113.129549,310.714286,1,6.0
C-3,4.742520,1200.416667,5693.000297,5693.000297,14.25,399.508793,279.656155,310.714286,1,6.0
C-3.3,4.742520,442.479167,2098.466389,2098.466389,19.00,110.445599,77.311920,310.714286,1,6.0
C-4,4.742520,594.625000,2820.021078,2820.021078,29.00,97.242106,68.069474,310.714286,1,6.0
C-4.2,4.742520,988.250000,4686.795594,4686.795594,29.00,161.613641,113.129549,310.714286,1,6.0
C-5,4.742520,747.104167,3543.156607,3543.156607,29.00,122.177814,85.524470,310.714286,1,6.0
C-5.5,4.742520,703.166667,3334.782125,3334.782125,29.00,114.992487,80.494741,310.714286,1,6.0
C-5.9,4.570116,773.333333,3534.223243,3534.223243,29.00,121.869767,85.308837,310.714286,1,6.0


,$w_{trib}$ [psf],$A$ [ft$^2$],$F_x$ [lbf],"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [lbf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,,,,
C-2.1,6.121092,1705.854167,10441.691088,18531.738943,29.00,639.025481,447.317837,460.714286,1,4.0
C-2.9,6.121092,988.250000,6049.169630,10735.965224,29.00,370.205697,259.143988,310.714286,1,6.0
C-3,6.121092,1200.416667,7347.861415,13040.861712,14.25,915.148190,640.603733,769.642857,1,2.0
C-3.3,6.121092,442.479167,2708.455894,4806.922282,19.00,252.995910,177.097137,310.714286,1,6.0
C-4,6.121092,594.625000,3639.754608,6459.775685,29.00,222.750886,155.925620,310.714286,1,6.0
C-4.2,6.121092,988.250000,6049.169630,10735.965224,29.00,370.205697,259.143988,310.714286,1,6.0
C-5,6.121092,747.104167,4573.093686,8116.250293,29.00,279.870700,195.909490,310.714286,1,6.0
C-5.5,6.121092,703.166667,4304.148186,7638.930311,29.00,263.411390,184.387973,310.714286,1,6.0
C-5.9,5.898574,773.333333,4561.563542,8095.786785,29.00,279.165062,195.415543,310.714286,1,6.0


,$w_{trib}$ [psf],$A$ [ft$^2$],$F_x$ [lbf],"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [lbf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,,,,
C-2.1,4.577796,1705.854167,7809.051706,26340.790649,29.00,908.303126,635.812188,769.642857,1,2.0
C-2.9,4.577796,988.250000,4524.006506,15259.971730,29.00,526.205922,368.344145,460.714286,1,4.0
C-3,4.577796,1200.416667,5495.262140,18536.123853,14.25,1300.780621,910.546435,921.428571,2,4.0
C-3.3,4.577796,442.479167,2025.579184,6832.501467,19.00,359.605340,251.723738,310.714286,1,6.0
C-4,4.577796,594.625000,2722.071711,9181.847397,29.00,316.615427,221.630799,310.714286,1,6.0
C-4.2,4.577796,988.250000,4524.006506,15259.971730,29.00,526.205922,368.344145,460.714286,1,4.0
C-5,4.577796,747.104167,3420.090170,11536.340463,29.00,397.804844,278.463390,310.714286,1,6.0
C-5.5,4.577796,703.166667,3218.953276,10857.883587,29.00,374.409779,262.086845,310.714286,1,6.0
C-5.9,4.411380,773.333333,3411.467095,11507.253880,29.00,396.801858,277.761301,310.714286,1,6.0


,$w_{trib}$ [psf],$A$ [ft$^2$],$F_x$ [lbf],"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [lbf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,,,,
C-2.1,3.184553,1705.854167,5432.383796,31773.174444,29.00,1095.626705,766.938693,769.642857,1,2.0
C-2.9,3.184553,988.250000,3147.134961,18407.106691,29.00,634.727817,444.309472,460.714286,1,4.0
C-3,3.184553,1200.416667,3822.791054,22358.914907,14.25,1569.046660,1098.332662,1200.000000,2,3.0
C-3.3,3.184553,442.479167,1409.098563,8241.600030,19.00,433.768423,303.637896,310.714286,1,6.0
C-4,3.184553,594.625000,1893.615104,11075.462500,29.00,381.912500,267.338750,310.714286,1,6.0
C-4.2,3.184553,988.250000,3147.134961,18407.106691,29.00,634.727817,444.309472,460.714286,1,4.0
C-5,3.184553,747.104167,2379.193162,13915.533625,29.00,479.845987,335.892191,460.714286,1,4.0
C-5.5,3.184553,703.166667,2239.271844,13097.155431,29.00,451.626049,316.138235,460.714286,1,4.0
C-5.9,3.068786,773.333333,2373.194501,13880.448380,29.00,478.636151,335.045306,460.714286,1,4.0


,$w_{trib}$ [psf],$A$ [ft$^2$],$F_x$ [lbf],"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [lbf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,,,,
C-2.1,1.825241,1705.854167,3113.595212,34886.769656,29.00,1202.992057,842.094440,921.428571,2,4.0
C-2.9,1.825241,988.250000,1803.794561,20210.901252,29.00,696.927629,487.849341,600.000000,1,3.0
C-3,1.825241,1200.416667,2191.049891,24549.964797,14.25,1722.804547,1205.963183,1539.285714,2,2.0
C-3.3,1.825241,442.479167,807.631181,9049.231210,19.00,476.275327,333.392729,460.714286,1,4.0
C-4,1.825241,594.625000,1085.334016,12160.796516,29.00,419.337811,293.536468,310.714286,1,6.0
C-4.2,1.825241,988.250000,1803.794561,20210.901252,29.00,696.927629,487.849341,600.000000,1,3.0
C-5,1.825241,747.104167,1363.645264,15279.178889,29.00,526.868238,368.807766,460.714286,1,4.0
C-5.5,1.825241,703.166667,1283.448731,14380.604162,29.00,495.882902,347.118032,460.714286,1,4.0
C-5.9,1.758888,773.333333,1360.207104,15240.655485,29.00,525.539844,367.877891,460.714286,1,4.0


#### East-West Direction

Shear wall shear forces are tabulated in \autoref{tab:shear_wall_forces_roof_ew} through \autoref{tab:shear_wall_forces_level2_ew}.

In [51]:
# Wall tributary widths and lengths
t_w__CB = l_CA__CB + l_CB__CC / 2
t_w__CC = l_CB__CC / 2 + l_CC__CD
t_w__CX = l_CW__CX + l_CX__CY / 2
t_w__CY = l_CX__CY / 2 + l_CY__CZ
t_w__CW6 = t_w__CX

t_l__CB__1 = fi(41, 5)
# t_l__CB__2 = fi(23, 5)
t_l__CB__3 = fi(28, 4)
t_l__CB__4 = fi(23, 1)
t_l__CC__1 = fi(77, 4)
t_l__CC__2 = fi(31, 7)
t_l__CC__3 = fi(17, 6)

t_l__CX__1 = fi(13, 10)
t_l__CX__2 = fi(21)
t_l__CW6__1 = fi(49, 1)
t_l__CY__1 = fi(14, 1)
t_l__CY__2 = fi(16, 4)
t_l__CY__3 = fi(59, 9)

In [52]:
# Wall segment lengths
l_CB__1 = fi(23, 3)
# l_CB__2 = fi(12, 9)
l_CB__3 = fi(9, 11)
l_CB__4 = fi(18, 8)
l_CC__1 = fi(28)
l_CC__2 = fi(19, 4)
l_CC__3 = fi(12, 1)

l_CX__1 = fi(9, 10)
l_CX__2 = fi(18, 11)
l_CW6__1 = fi(19, 1)
l_CY__1 = fi(8, 10)
l_CY__2 = fi(11, 4)
l_CY__3 = fi(19, 7)

In [ ]:
flexible_ew.loc["Level 5", "tributary weight"] = float(f_p__roof)
flexible_ew.loc["Level 4", "tributary weight"] = float(f_p__5)
flexible_ew.loc["Level 3", "tributary weight"] = float(f_p__4)
flexible_ew.loc["Level 2", "tributary weight"] = float(f_p__3)
flexible_ew.loc["Level 1", "tributary weight"] = float(f_p__2)

# Nonparallel irregularity
angle_degrees = 23
angle = math.radians(angle_degrees)
angled_walls = ["C-B--C-2", "C-B--C-3.3", "C-B--C-4", "C-B--C-4.9", "C-C--C-2.5", "C-C--C-3.6", "C-C--C-5.2"]
wall_indices = flexible_ew.index.get_level_values("Wall").isin(angled_walls)
flexible_ew.loc[wall_indices, "tributary weight"] *= math.cos(angle) + 0.3 * math.sin(angle)

# Shear wall height is for the floor below
flexible_ew.loc["Level 5", "wall height"] = float(seismic_loads.loc["Level 5", "floor height"])
flexible_ew.loc["Level 4", "wall height"] = float(seismic_loads.loc["Level 4", "floor height"])
flexible_ew.loc["Level 3", "wall height"] = float(seismic_loads.loc["Level 3", "floor height"])
flexible_ew.loc["Level 2", "wall height"] = float(seismic_loads.loc["Level 2", "floor height"])
flexible_ew.loc["Level 1", "wall height"] = float(seismic_loads.loc["Level 1", "floor height"])

# Add wall tributary widths and wall lengths to df
wall_tributary = {
    "C-B--C-2": {"trib_width": float(t_w__CB), "trib_depth": t_l__CB__1, "wall_length": l_CB__1},
    "C-B--C-4": {"trib_width": float(t_w__CB), "trib_depth": t_l__CB__3, "wall_length": l_CB__3},
    "C-B--C-4.9": {"trib_width": float(t_w__CB), "trib_depth": t_l__CB__4, "wall_length": l_CB__4},
    "C-C--C-2.5": {"trib_width": float(t_w__CC), "trib_depth": t_l__CC__1, "wall_length": l_CC__1},
    "C-C--C-3.6": {"trib_width": float(t_w__CC), "trib_depth": t_l__CC__2, "wall_length": l_CC__2},
    "C-C--C-5.2": {"trib_width": float(t_w__CC), "trib_depth": t_l__CC__3, "wall_length": l_CC__3},
    # "C-X--C-5.8": {"trib_width": float(t_w__CX), "trib_depth": t_l__CX__1, "wall_length": l_CX__1},
    "C-X--C-6.5": {"trib_width": float(t_w__CX), "trib_depth": t_l__CX__2, "wall_length": l_CX__2},
    "C-W.6--C-7.3": {"trib_width": float(t_w__CW6), "trib_depth": t_l__CW6__1, "wall_length": l_CW6__1},
    # "C-Y--C-5.8": {"trib_width": float(t_w__CY), "trib_depth": t_l__CY__1, "wall_length": l_CY__1},
    "C-Y--C-6.5": {"trib_width": float(t_w__CY), "trib_depth": t_l__CY__2, "wall_length": l_CY__2},
    "C-Y--C-7.2": {"trib_width": float(t_w__CY), "trib_depth": t_l__CY__3, "wall_length": l_CY__3},
}

flexible_ew["wall length"] = flexible_ew.index.get_level_values("Wall").map(
    {key: value["wall_length"] for key, value in wall_tributary.items()}
)
flexible_ew["tributary width"] = flexible_ew.index.get_level_values("Wall").map(
    {key: value["trib_width"] for key, value in wall_tributary.items()}
)
flexible_ew["tributary depth"] = flexible_ew.index.get_level_values("Wall").map(
    {key: value["trib_depth"] for key, value in wall_tributary.items()}
)
flexible_ew["tributary area"] = flexible_ew["tributary depth"] * flexible_ew["tributary width"]

# Fix trib area of certain walls on all levels
flexible_ew.loc[flexible_ew.index.get_level_values("Wall") == "C-B--C-2", "tributary area"] += 100
flexible_ew.loc[flexible_ew.index.get_level_values("Wall") == "C-B--C-4.9", "tributary area"] -= 120
flexible_ew.loc[flexible_ew.index.get_level_values("Wall") == "C-C--C-5.2", "tributary area"] += 160
# flexible_ew.loc[flexible_ew.index.get_level_values("Wall") == "C-X--C-5.8", "tributary area"] -= 80.25
flexible_ew.loc[flexible_ew.index.get_level_values("Wall") == "C-X--C-6.5", "tributary area"] += 480 - 91
# flexible_ew.loc[flexible_ew.index.get_level_values("Wall") == "C-Y--C-5.8", "tributary area"] += 145
flexible_ew.loc[flexible_ew.index.get_level_values("Wall") == "C-Y--C-6.5", "tributary area"] += 480 + 131
flexible_ew.loc[flexible_ew.index.get_level_values("Wall") == "C-W.6--C-7.3", "tributary area"] += 35.5

# Ensure that the calculated areas from earlier and tributary areas match
trib_area = flexible_ew["tributary area"].sum() / len(flexible_ns.index.get_level_values("Level").unique())
assert np.isclose(trib_area, float(A_level)), (
    f"Shear wall tributary areas ({trib_area}) and level area ({float(A_level)}) do not match up.\n\
    Ensure you are including all areas in your calculations"
)

# Automatically extract the shear line by splitting at "_" and taking the first section
flexible_ew["line"] = flexible_ew.index.get_level_values("Wall").str.split("--").str[0]

# Get tributary area and wall length for the shear line
flexible_ew["line tributary area"] = flexible_ew.groupby(["Level", "line"])["tributary area"].transform("sum")
flexible_ew["line wall length"] = flexible_ew.groupby(["Level", "line"])["wall length"].transform("sum")

flexible_ew["line shear demand"] = flexible_ew["line tributary area"] * flexible_ew["tributary weight"]
flexible_ew["line unit shear"] = flexible_ew["line shear demand"] / flexible_ew["line wall length"]
flexible_ew["line cumulative shear demand"] = flexible_ew.groupby(level="Wall")["line shear demand"].cumsum()
flexible_ew["line cumulative unit shear"] = (
    flexible_ew["line cumulative shear demand"] / flexible_ew["line wall length"]
)

flexible_ew["shear demand"] = flexible_ew["line unit shear"] * flexible_ew["wall length"]
flexible_ew["cumulative shear demand"] = flexible_ew.groupby(level="Wall")["shear demand"].cumsum()
flexible_ew["unit shear demand"] = flexible_ew["cumulative shear demand"] / flexible_ew["wall length"]
flexible_ew["adjusted unit shear demand"] = flexible_ew["unit shear demand"] * LOAD_FACTOR_SEISMIC_ASD

flexible_ew["sheathed sides"] = 1
flexible_ew["aspect ratio"] = flexible_ew["wall height"] / flexible_ew["wall length"]
flexible_ew["aspect ratio factor"] = np.where(
    flexible_ew["aspect ratio"] > 2, 1.25 - 0.125 * flexible_ew["wall height"] / flexible_ew["wall length"], 1
)

#### Check aspect ratios

We will also check the aspect ratios of the E-W walls.

In [54]:
%%render
ratio_max = check_value(flexible_ew["aspect ratio"].max(), 2, "<=")

<IPython.core.display.Latex object>

In [55]:
if "OK" in ratio_max:
    display_text(r"$\therefore$ All aspect ratios are OK")
else:
    display_text("The aspect ratio knockdown factor must be applied.")

'$\\therefore$ All aspect ratios are OK'

In [56]:
# using sheathing from the NS calc in memory

flexible_ew[["adjusted unit shear capacity", "nail spacing", "shear stiffness", "shear dcr"]] = (
    get_sheathing_properties(
        flexible_ew[["adjusted unit shear demand", "sheathed sides"]],
        sheathing=sheathing,
    )
)

# Assign 2-sided sheathing to walls requiring it, then recalc capacity for just those walls (more efficient than passing the full wall dataframe)
if (flexible_ew["shear dcr"] > shear_dcr_check_value).any():
    flexible_ew.loc[flexible_ew["shear dcr"] > shear_dcr_check_value, "sheathed sides"] = 2
    flexible_ew.loc[
        flexible_ew["shear dcr"] > shear_dcr_check_value,
        ["adjusted unit shear capacity", "nail spacing", "shear stiffness", "shear dcr"],
    ] = get_sheathing_properties(
        flexible_ew.loc[
            flexible_ew["shear dcr"] > shear_dcr_check_value,
            ["adjusted unit shear demand", "sheathed sides"],
        ],
        sheathing=sheathing,
    )

In [57]:
# Update dataframe
flexible.update(flexible_ew)

In [58]:
# Rename columns for display with LaTeX formatting for units and subscripts/superscripts
column_name_map = {
    "tributary weight": "$W_{trib}$ [psf]",
    "tributary area": "$A$ [ft$^2$]",
    "shear demand": "$F_x$ [lbf]",
    "cumulative shear demand": "$F_{x,cum}$ [lbf]",
    "wall length": "$l_{wall}$ [ft]",
    "unit shear demand": "$v$ [plf]",
    "adjusted unit shear demand": "$0.7 v$ [plf]",
    "adjusted unit shear capacity": "$v_{cap}$ [plf]",
    "sheathed sides": "sides",
    "nail spacing": "$s_{nail}$ [in]",
}

In [ ]:
display_table(
    dataframe=flexible_ew,
    levels="Level 5",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Shear wall forces EW - Level 5",
    label="tab:shear_wall_forces_roof_ew",
)
display_table(
    dataframe=flexible_ew,
    levels="Level 4",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Shear wall forces EW - Level 4",
    label="tab:shear_wall_forces_level4_ew",
)
display_table(
    dataframe=flexible_ew,
    levels="Level 3",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Shear wall forces EW - Level 3",
    label="tab:shear_wall_forces_level3_ew",
)
display_table(
    dataframe=flexible_ew,
    levels="Level 2",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Shear wall forces EW - Level 2",
    label="tab:shear_wall_forces_level2_ew",
)
display_table(
    dataframe=flexible_ew,
    levels="Level 1",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Shear wall forces EW - Level 1",
    label="tab:shear_wall_forces_level1_ew",
)

,$W_{trib}$ [psf],$A$ [ft$^2$],$F_x$ [lbf],"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [plf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,,,,
C-B--C-2,4.742520,1487.458333,6573.092976,6573.092976,23.250000,282.713676,197.899573,310.714286,1,6.0
C-B--C-4,4.742520,949.166667,2803.577291,2803.577291,9.916667,282.713676,197.899573,310.714286,1,6.0
C-B--C-4.9,4.742520,653.291667,5277.321960,5277.321960,18.666667,282.713676,197.899573,310.714286,1,6.0
C-W.6--C-7.3,4.570116,1679.791667,7676.843214,7676.843214,19.083333,402.279994,281.595996,310.714286,1,6.0
C-X--C-6.5,4.570116,1092.500000,4992.852017,4992.852017,18.916667,263.939314,184.757520,310.714286,1,6.0
C-C--C-2.5,4.742520,2590.666667,9822.311421,9822.311421,28.000000,350.796836,245.557786,310.714286,1,6.0
C-C--C-3.6,4.742520,1058.041667,6782.072172,6782.072172,19.333333,350.796836,245.557786,310.714286,1,6.0
C-C--C-5.2,4.742520,746.250000,4238.795107,4238.795107,12.083333,350.796836,245.557786,310.714286,1,6.0
C-Y--C-6.5,4.570116,1158.166667,5293.594821,5293.594821,11.333333,467.081896,326.957327,460.714286,1,4.0


,$W_{trib}$ [psf],$A$ [ft$^2$],$F_x$ [lbf],"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [plf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,,,,
C-B--C-2,6.121092,1487.458333,8483.782493,15056.875470,23.250000,647.607547,453.325283,460.714286,1,4.0
C-B--C-4,6.121092,949.166667,3618.530884,6422.108175,9.916667,647.607547,453.325283,460.714286,1,4.0
C-B--C-4.9,6.121092,653.291667,6811.352253,12088.674212,18.666667,647.607547,453.325283,460.714286,1,4.0
C-W.6--C-7.3,5.898574,1679.791667,9908.374687,17585.217901,19.083333,921.496135,645.047294,769.642857,1,2.0
C-X--C-6.5,5.898574,1092.500000,6444.191598,11437.043615,18.916667,604.601425,423.220997,460.714286,1,4.0
C-C--C-2.5,6.121092,2590.666667,12677.495051,22499.806473,28.000000,803.564517,562.495162,600.000000,1,3.0
C-C--C-3.6,6.121092,1058.041667,8753.508488,15535.580660,19.333333,803.564517,562.495162,600.000000,1,3.0
C-C--C-5.2,6.121092,746.250000,5470.942805,9709.737912,12.083333,803.564517,562.495162,600.000000,1,3.0
C-Y--C-6.5,5.898574,1158.166667,6832.355366,12125.950186,11.333333,1069.936781,748.955747,769.642857,1,2.0


,$W_{trib}$ [psf],$A$ [ft$^2$],$F_x$ [lbf],"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [plf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,,,,
C-B--C-2,4.577796,1487.458333,6344.786069,21401.661539,23.250000,920.501572,644.351100,769.642857,1,2.0
C-B--C-4,4.577796,949.166667,2706.199076,9128.307251,9.916667,920.501572,644.351100,769.642857,1,2.0
C-B--C-4.9,4.577796,653.291667,5094.021790,17182.696002,18.666667,920.501572,644.351100,769.642857,1,2.0
C-W.6--C-7.3,4.411380,1679.791667,7410.199133,24995.417034,19.083333,1309.803513,916.862459,921.428571,2,4.0
C-X--C-6.5,4.411380,1092.500000,4819.432501,16256.476117,18.916667,859.373187,601.561231,769.642857,1,2.0
C-C--C-2.5,4.577796,2590.666667,9481.147596,31980.954069,28.000000,1142.176931,799.523852,921.428571,2,4.0
C-C--C-3.6,4.577796,1058.041667,6546.506674,22082.087333,19.333333,1142.176931,799.523852,921.428571,2,4.0
C-C--C-5.2,4.577796,746.250000,4091.566671,13801.304583,12.083333,1142.176931,799.523852,921.428571,2,4.0
C-Y--C-6.5,4.411380,1158.166667,5109.729437,17235.679623,11.333333,1520.795261,1064.556683,1200.000000,2,3.0


,$W_{trib}$ [psf],$A$ [ft$^2$],$F_x$ [lbf],"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [plf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,,,,
C-B--C-2,3.184553,1487.458333,4413.764222,25815.425761,23.250000,1110.340893,777.238625,921.428571,2,4.0
C-B--C-4,3.184553,949.166667,1882.573270,11010.880522,9.916667,1110.340893,777.238625,921.428571,2,4.0
C-B--C-4.9,3.184553,653.291667,3543.667332,20726.363335,18.666667,1110.340893,777.238625,921.428571,2,4.0
C-W.6--C-7.3,3.068786,1679.791667,5154.921136,30150.338171,19.083333,1579.930384,1105.951269,1200.000000,2,3.0
C-X--C-6.5,3.068786,1092.500000,3352.648696,19609.124813,18.916667,1036.605717,725.624002,769.642857,1,2.0
C-C--C-2.5,3.184553,2590.666667,6595.580936,38576.535006,28.000000,1377.733393,964.413375,1200.000000,2,3.0
C-C--C-3.6,3.184553,1058.041667,4554.091599,26636.178932,19.333333,1377.733393,964.413375,1200.000000,2,3.0
C-C--C-5.2,3.184553,746.250000,2846.307249,16647.611833,12.083333,1377.733393,964.413375,1200.000000,2,3.0
C-Y--C-6.5,3.068786,1158.166667,3554.594391,20790.274014,11.333333,1834.435942,1284.105160,1539.285714,2,2.0


,$W_{trib}$ [psf],$A$ [ft$^2$],$F_x$ [lbf],"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [plf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,,,,
C-B--C-2,1.825241,1487.458333,2529.768820,28345.194581,23.250000,1219.148154,853.403708,921.428571,2,4.0
C-B--C-4,1.825241,949.166667,1079.005339,12089.885861,9.916667,1219.148154,853.403708,921.428571,2,4.0
C-B--C-4.9,1.825241,653.291667,2031.068874,22757.432208,18.666667,1219.148154,853.403708,921.428571,2,4.0
C-W.6--C-7.3,1.758888,1679.791667,2954.566240,33104.904411,19.083333,1734.754816,1214.328371,1539.285714,2,2.0
C-X--C-6.5,1.758888,1092.500000,1921.585683,21530.710496,18.916667,1138.187339,796.731137,921.428571,2,4.0
C-C--C-2.5,1.825241,2590.666667,3780.286886,42356.821891,28.000000,1512.743639,1058.920547,1200.000000,2,3.0
C-C--C-3.6,1.825241,1058.041667,2610.198088,29246.377020,19.333333,1512.743639,1058.920547,1200.000000,2,3.0
C-C--C-5.2,1.825241,746.250000,1631.373805,18278.985638,12.083333,1512.743639,1058.920547,1200.000000,2,3.0
C-Y--C-6.5,1.758888,1158.166667,2037.331766,22827.605781,11.333333,2014.200510,1409.940357,1539.285714,2,2.0


### Rigid Diaphragm Assumption

With a rigid diaphragm assumption, we now must calculate the following in each direction:

1. The relative stiffnesses of the shear walls to obtain a center of rigidity.
2. The center of mass of the diaphragm with the center of rigidity to obtain an eccentricity.

The relative stiffnesses of the shear walls can be calculated by using the deflection equation from NDS SDPWS Sec. 4.3.4

\begin{align*}
k &= \frac{F}{\delta},\, \text{assume }F = \text{induced load demand in the shear wall} \\
\delta_{sw} &= \frac{8vh^3}{EAb} + \frac{vh}{1000 G_a} + \frac{h \Delta_A}{b}\quad \text{(SDPWS Eq. 4.3-1)}
\end{align*}

where

- $b =$ the shear wall length [ft]
- $\Delta_A =$ vertical deformation of the wall overturning anchorage system + vertical compression deformation [in]
- $E =$ modulus of elasticity of end posts [psi]
- $A =$ area of end post cross-section [in$^2$]
- $G_a =$ apparent shear wall shear stiffness from nail slip and panel shear deformation [kip/in]
- $h =$ shear wall height [ft]
- $v =$ unit shear force induced by the design load [plf]
- $\delta_{sw} =$ maximum shear wall deflection determined by elastic analysis [in]

Note that for the following calculations, we will conservatively use the unit shear force induced by the nominal load, not the design load, for the value of $v$. This is because the stiffnesses obtained from this use will allow us to use the stiffnesses for deflection limits and story drift calculations

We will assume that the end posts are (2) 2x4 Douglas Fir No. 2; this means:

In [60]:
%%render
E_endpost = 1_600_000 * psi
A_endpost = 2 * 5.25 * inch**2

<IPython.core.display.Latex object>

We will assume a continuous tiedown rod system. A conservative first-pass value for $\Delta_A$ can be:

In [61]:
%%render params
Delta_A = 0.25 * inch

<IPython.core.display.Latex object>

#### Shear Wall Rigidities

With the deflections of each shear wall, along with the force demand of each shear wall, we can obtain the stiffnesses (and therefore, relative stiffnesses) of all the walls.

Each relative stiffness is relative to the level. The sum of all relative stiffnesses on each level is $1.0$.

\begin{align*}
k &= \frac{F_x}{\delta_{sw}} \\
R &= \frac{k_{wall}}{\sum {k_{wall}}}
\end{align*}

##### North-South Direction

Rigidities in the N-S direction can be found in \autoref{tab:wall_rigidities_ns_roof} through \autoref{tab:wall_rigidities_ns_level2}.

In [ ]:
rigid_ns["unit shear demand"] = flexible_ns["unit shear demand"]
rigid_ns["wall length"] = flexible_ns["wall length"]
rigid_ns["wall height"] = flexible_ns["wall height"]
rigid_ns["shear stiffness"] = flexible_ns["shear stiffness"]
rigid_ns["sheathed sides"] = flexible_ns["sheathed sides"]
rigid_ns["nail spacing"] = flexible_ns["nail spacing"]
rigid_ns["end post youngs modulus"] = float(E_endpost)
rigid_ns["end post area"] = float(A_endpost)
rigid_ns["Delta_A"] = float(Delta_A)

rigid_ns["delta_sw"] = (
    8
    * rigid_ns["unit shear demand"]
    * rigid_ns["wall height"] ** 3
    / (rigid_ns["end post youngs modulus"] * rigid_ns["end post area"] * rigid_ns["wall length"])
    + rigid_ns["unit shear demand"]
    * rigid_ns["wall height"]
    / (1000 * rigid_ns["shear stiffness"] * rigid_ns["sheathed sides"])
    + rigid_ns["wall height"] * rigid_ns["Delta_A"] / rigid_ns["wall length"]
)

In [63]:
rigid_ns["cumulative shear demand"] = flexible_ns["cumulative shear demand"]
rigid_ns["wall stiffness"] = rigid_ns["cumulative shear demand"] / rigid_ns["delta_sw"]
rigid_ns["total level stiffness"] = rigid_ns.groupby("Level")["wall stiffness"].transform("sum")
rigid_ns["total level cumulative shear"] = rigid_ns.groupby("Level")["cumulative shear demand"].transform("sum")
rigid_ns["relative wall stiffness"] = rigid_ns["wall stiffness"] / rigid_ns["total level stiffness"]
rigid_ns["direct shear force"] = rigid_ns["total level cumulative shear"] * rigid_ns["relative wall stiffness"]

In [64]:
column_name_map = {
    "cumulative shear demand": "$F_x$ [lbf]",
    "delta_sw": r"$\delta_{sw}$ [in]",
    "wall stiffness": "$k$ [lbf/in]",
    "relative wall stiffness": "$R$ [-]",
}

In [ ]:
display_table(
    dataframe=rigid_ns,
    levels="Level 5",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Wall rigidities NS - Level 5",
    label="tab:wall_rigidities_ns_level5",
)
display_table(
    dataframe=rigid_ns,
    levels="Level 4",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Wall rigidities NS - Level 4",
    label="tab:wall_rigidities_ns_level4",
)
display_table(
    dataframe=rigid_ns,
    levels="Level 3",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Wall rigidities NS - Level 3",
    label="tab:wall_rigidities_ns_level3",
)
display_table(
    dataframe=rigid_ns,
    levels="Level 2",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Wall rigidities NS - Level 2",
    label="tab:wall_rigidities_ns_level2",
)
display_table(
    dataframe=rigid_ns,
    levels="Level 1",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Wall rigidities NS - Level 1",
    label="tab:wall_rigidities_ns_level1",
)

,$F_x$ [lbf],$\delta_{sw}$ [in],$k$ [lbf/in],$R$ [-]
Wall,,,,
C-2.1,8090.047854,0.305045,26520.799552,0.109343
C-2.9,4686.795594,0.214800,21819.394625,0.089960
C-3,5693.000297,0.499297,11402.036347,0.047010
C-3.3,2098.466389,0.224196,9359.943935,0.038590
C-4,2820.021078,0.165297,17060.301482,0.070338
C-4.2,4686.795594,0.214800,21819.394625,0.089960
C-5,3543.156607,0.184473,19206.908116,0.079189
C-5.5,3334.782125,0.178947,18635.538127,0.076833
C-5.9,3534.223243,0.184236,19183.115698,0.079091


,$F_x$ [lbf],$\delta_{sw}$ [in],$k$ [lbf/in],$R$ [-]
Wall,,,,
C-2.1,18531.738943,0.497356,37260.478276,0.107919
C-2.9,10735.965224,0.375209,28613.322884,0.082874
C-3,13040.861712,0.637397,20459.547895,0.059258
C-3.3,4806.922282,0.335245,14338.533755,0.041529
C-4,6459.775685,0.261815,24673.092579,0.071462
C-4.2,10735.965224,0.375209,28613.322884,0.082874
C-5,8116.250293,0.305740,26546.230114,0.076887
C-5.5,7638.930311,0.293083,26064.062639,0.075490
C-5.9,8095.786785,0.305198,26526.379327,0.076829


,$F_x$ [lbf],$\delta_{sw}$ [in],$k$ [lbf/in],$R$ [-]
Wall,,,,
C-2.1,26340.790649,0.522443,50418.501132,0.125693
C-2.9,15259.971730,0.425529,35861.159828,0.089402
C-3,18536.123853,1.037954,17858.337054,0.044521
C-3.3,6832.501467,0.418295,16334.162302,0.040721
C-4,9181.847397,0.333997,27490.788686,0.068534
C-4.2,15259.971730,0.425529,35861.159828,0.089402
C-5,11536.340463,0.396433,29100.383259,0.072547
C-5.5,10857.883587,0.378442,28691.042165,0.071527
C-5.9,11507.253880,0.395661,29083.597882,0.072505


,$F_x$ [lbf],$\delta_{sw}$ [in],$k$ [lbf/in],$R$ [-]
Wall,,,,
C-2.1,31773.174444,0.611521,51957.617658,0.117834
C-2.9,18407.106691,0.494620,37214.619543,0.084399
C-3,22358.914907,1.112013,20106.710203,0.045600
C-3.3,8241.600030,0.476069,17311.768899,0.039261
C-4,11075.462500,0.384211,28826.492220,0.065375
C-4.2,18407.106691,0.494620,37214.619543,0.084399
C-5,13915.533625,0.396014,35139.001740,0.079692
C-5.5,13097.155431,0.378048,34644.203088,0.078569
C-5.9,13880.448380,0.395244,35118.711814,0.079646


,$F_x$ [lbf],$\delta_{sw}$ [in],$k$ [lbf/in],$R$ [-]
Wall,,,,
C-2.1,34886.769656,1.120298,31140.623979,0.089646
C-2.9,20210.901252,0.639721,31593.316822,0.090949
C-3,24549.964797,1.389699,17665.669454,0.050855
C-3.3,9049.231210,0.585219,15462.981521,0.044514
C-4,12160.796516,0.537682,22617.074337,0.065109
C-4.2,20210.901252,0.639721,31593.316822,0.090949
C-5,15279.178889,0.556060,27477.558428,0.079101
C-5.5,14380.604162,0.530202,27122.854204,0.078080
C-5.9,15240.655485,0.554952,27463.029842,0.079059


##### East-West Direction

Rigidities in the E-W direction can be found in \autoref{tab:wall_rigidities_ew_roof}, \autoref{tab:wall_rigidities_ew_level4}, \autoref{tab:wall_rigidities_ew_level3}, \autoref{tab:wall_rigidities_ew_level2},

In [ ]:
rigid_ew["unit shear demand"] = flexible_ew["unit shear demand"]
rigid_ew["wall length"] = flexible_ew["wall length"]
rigid_ew["wall height"] = flexible_ew["wall height"]
rigid_ew["shear stiffness"] = flexible_ew["shear stiffness"]
rigid_ew["sheathed sides"] = flexible_ew["sheathed sides"]
rigid_ew["nail spacing"] = flexible_ew["nail spacing"]
rigid_ew["end post youngs modulus"] = float(E_endpost)
rigid_ew["end post area"] = float(A_endpost)
rigid_ew["Delta_A"] = float(Delta_A)

rigid_ew["delta_sw"] = (
    8
    * rigid_ew["unit shear demand"]
    * rigid_ew["wall height"] ** 3
    / (rigid_ew["end post youngs modulus"] * rigid_ew["end post area"] * rigid_ew["wall length"])
    + rigid_ew["unit shear demand"]
    * rigid_ew["wall height"]
    / (1000 * rigid_ew["shear stiffness"] * rigid_ew["sheathed sides"])
    + rigid_ew["wall height"] * rigid_ew["Delta_A"] / rigid_ew["wall length"]
)

In [67]:
rigid_ew["cumulative shear demand"] = flexible_ew["cumulative shear demand"]
rigid_ew["wall stiffness"] = rigid_ew["cumulative shear demand"] / rigid_ew["delta_sw"]
rigid_ew["total level stiffness"] = rigid_ew.groupby("Level")["wall stiffness"].transform("sum")
rigid_ew["total level cumulative shear"] = rigid_ew.groupby("Level")["cumulative shear demand"].transform("sum")
rigid_ew["relative wall stiffness"] = rigid_ew["wall stiffness"] / rigid_ew["total level stiffness"]
rigid_ew["direct shear force"] = rigid_ew["total level cumulative shear"] * rigid_ew["relative wall stiffness"]

In [68]:
column_name_map = {
    "cumulative shear demand": "$F_x$ [lbf]",
    "delta_sw": r"$\delta_{sw}$ [in]",
    "wall stiffness": "$k$ [lbf/in]",
    "relative wall stiffness": "$R$ [-]",
}

In [ ]:
display_table(
    dataframe=rigid_ew,
    levels="Level 5",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Wall rigidities EW - Level 5",
    label="tab:wall_rigidities_ew_level5",
)
display_table(
    dataframe=rigid_ew,
    levels="Level 4",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Wall rigidities EW - Level 4",
    label="tab:wall_rigidities_ew_level4",
)
display_table(
    dataframe=rigid_ew,
    levels="Level 3",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Wall rigidities EW - Level 3",
    label="tab:wall_rigidities_ew_level3",
)
display_table(
    dataframe=rigid_ew,
    levels="Level 2",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Wall rigidities EW - Level 2",
    label="tab:wall_rigidities_ew_level2",
)
display_table(
    dataframe=rigid_ew,
    levels="Level 1",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Wall rigidities EW - Level 1",
    label="tab:wall_rigidities_ew_level1",
)

,$F_x$ [lbf],$\delta_{sw}$ [in],$k$ [lbf/in],$R$ [-]
Wall,,,,
C-B--C-2,6573.092976,0.331642,19819.872720,0.128259
C-B--C-4,2803.577291,0.492457,5693.043317,0.036841
C-B--C-4.9,5277.321960,0.361009,14618.250048,0.094598
C-W.6--C-7.3,7676.843214,0.450885,17026.165789,0.110180
C-X--C-6.5,4992.852017,0.344412,14496.723278,0.093811
C-C--C-2.5,9822.311421,0.363754,27002.625509,0.174740
C-C--C-3.6,6782.072172,0.408876,16587.123138,0.107339
C-C--C-5.2,4238.795107,0.496343,8540.059034,0.055265
C-Y--C-6.5,5293.594821,0.542828,9751.880024,0.063107


,$F_x$ [lbf],$\delta_{sw}$ [in],$k$ [lbf/in],$R$ [-]
Wall,,,,
C-B--C-2,15056.875470,0.528251,28503.276561,0.119233
C-B--C-4,6422.108175,0.700698,9165.299307,0.038340
C-B--C-4.9,12088.674212,0.559743,21596.846727,0.090342
C-W.6--C-7.3,17585.217901,0.584856,30067.583929,0.125776
C-X--C-6.5,11437.043615,0.529815,21586.842935,0.090300
C-C--C-2.5,22499.806473,0.553645,40639.388256,0.170000
C-C--C-3.6,15535.580660,0.602763,25773.946720,0.107816
C-C--C-5.2,9709.737912,0.697976,13911.284114,0.058193
C-Y--C-6.5,12125.950186,0.772108,15704.982983,0.065696


,$F_x$ [lbf],$\delta_{sw}$ [in],$k$ [lbf/in],$R$ [-]
Wall,,,,
C-B--C-2,21401.661539,0.554957,38564.540243,0.149780
C-B--C-4,9128.307251,0.736104,12400.840452,0.048163
C-B--C-4.9,17182.696002,0.588038,29220.406561,0.113489
C-W.6--C-7.3,24995.417034,0.984386,25391.873999,0.098619
C-X--C-6.5,16256.476117,0.556132,29231.325053,0.113531
C-C--C-2.5,31980.954069,0.821699,38920.530107,0.151163
C-C--C-3.6,22082.087333,0.873805,25271.187457,0.098150
C-C--C-5.2,13801.304583,0.974811,14157.935899,0.054988
C-Y--C-6.5,17235.679623,1.146028,15039.490400,0.058412


,$F_x$ [lbf],$\delta_{sw}$ [in],$k$ [lbf/in],$R$ [-]
Wall,,,,
C-B--C-2,25815.425761,0.825028,31290.369370,0.117376
C-B--C-4,11010.880522,1.012227,10877.881452,0.040805
C-B--C-4.9,20726.363335,0.859214,24122.482175,0.090488
C-W.6--C-7.3,30150.338171,1.056313,28543.006991,0.107070
C-X--C-6.5,19609.124813,0.642207,30533.950706,0.114539
C-C--C-2.5,38576.535006,0.882253,43725.020314,0.164021
C-C--C-3.6,26636.178932,0.936438,28444.143911,0.106699
C-C--C-5.2,16647.611833,1.041474,15984.671446,0.059961
C-Y--C-6.5,20790.274014,1.158304,17948.893174,0.067330


,$F_x$ [lbf],$\delta_{sw}$ [in],$k$ [lbf/in],$R$ [-]
Wall,,,,
C-B--C-2,28345.194581,1.174743,24128.841073,0.118841
C-B--C-4,12089.885861,1.452520,8323.388156,0.040995
C-B--C-4.9,22757.432208,1.225470,18570.369744,0.091464
C-W.6--C-7.3,33104.904411,1.301586,25434.290294,0.125270
C-X--C-6.5,21530.710496,1.152763,18677.488988,0.091991
C-C--C-2.5,42356.821891,1.258678,33651.838891,0.165744
C-C--C-3.6,29246.377020,1.341086,21807.981021,0.107410
C-C--C-5.2,18278.985638,1.500831,12179.242907,0.059986
C-Y--C-6.5,22827.605781,1.688264,13521.344804,0.066596


#### Center of Rigidity and Center of Mass

We can find the center of rigidity by taking the weighted sum of all wall stiffnesses and their locations relative to a datum. In this case, we will use the north-west-most corner of the building (i.e., the intersection of lines B-1 and B-A).

$$
\bar{x}_r = \frac{\sum k_{ns} x}{\sum k_{ns}},\, \bar{y}_r = \frac{\sum k_{ew} y}{\sum k_{ew}}
$$

Note: each wall is assumed to have zero rigidity in the direction perpendicular to them.

The top floor has a higher concentration of load on the N-E corner, but this area is relatively small and is only present on the top floor. We will assume the center of mass is the same as for a uniform floor weight.

All dimensions are measured from an origin at the S-E corner of the building (i.e., the intersection of C-9 and C-Z).

In [70]:
%%render params
X_plan = fi(221, 6)  # ft
Y_plan = fi(115, 6)  # ft
CM_x = fi(102)  # ft
CM_y = fi(43, 3)  # ft

<IPython.core.display.Latex object>

An accidental torsion eccentricity of 5% must be added/subtracted.

In [71]:
%%render
CM_x__a__plus = CM_x + X_plan * 0.05
CM_x__a__minus = CM_x - X_plan * 0.05
CM_y__a__plus = CM_y + Y_plan * 0.05
CM_y__a__minus = CM_y - Y_plan * 0.05

<IPython.core.display.Latex object>

The eccentricity used is determined by the location of the wall in relation to the center of rigidity. Since the wall rotates about the center of rigidity, we want to position the center of mass to ensure each wall is getting the maximum possible force to be conservative. If the center of mass is positioned on the opposite side of the center of rigidity relative to the wall in question, that mass will _decrease_ the shear experienced by that wall.

This means we always want the center of mass to be on the same side of the center of rigidity as the wall. The eccentricities shown in the following tables are based on the worst-case scenario. This is not immediately obvious due to all torques being given a positive value. 

How the center of rigidity, center of mass, and eccentricity interact are illustrated in \autoref{fig:rigid_diaphragm_moments}. The moments generated by the rotation of the center of mass around the center of rigidity differ based on the chosen eccentricity.

![Moments applied by a rigid diaphragm](images/rigid_diaphragm_moments.png){width=50% #fig:rigid_diaphragm_moments}

$$
d_x = \left| x - \text{CR}_x \right|\text{ and }\, d_y = \left| y - \text{CR}_y \right|
$$

In [72]:
# Update dataframe
rigid = rigid.reindex(columns=rigid_ns.columns)
rigid.update(rigid_ns)
rigid.update(rigid_ew)

In [73]:
# Location of wall lines of rigidity
wall_rigidity_loc = {
    # N-S
    "C-2.1": {"x": fi(182, 8), "y": 0},
    "C-2.9": {"x": fi(160), "y": 0},
    "C-3": {"x": fi(145, 1), "y": 0},
    "C-3.3": {"x": fi(137, 1), "y": 0},
    "C-4": {"x": fi(118, 9), "y": 0},
    "C-4.2": {"x": fi(120, 4), "y": 0},
    "C-5": {"x": fi(102, 2), "y": 0},
    "C-5.5": {"x": fi(104, 11), "y": 0},
    "C-5.9": {"x": l_C59__C7 + l_C7__C83 + l_C83__C9, "y": 0},
    "C-6": {"x": l_C6__C7 + l_C7__C8 + l_C8__C9, "y": 0},
    "C-7": {"x": l_C7__C8 + l_C8__C9, "y": 0},
    "C-7.3": {"x": l_C73__C8 + l_C8__C9, "y": 0},
    # "C-8--C-W": {"x": l_C8__C9, "y": 0},
    "C-8--C-W.6": {"x": l_C8__C9, "y": 0},
    "C-8.1": {"x": l_C81__C83 + l_C83__C9, "y": 0},
    "C-8.3": {"x": l_C83__C9, "y": 0},
    # E-W
    "C-B--C-2": {"x": 0, "y": fi(82, 4)},
    "C-B--C-4": {"x": 0, "y": fi(63, 4)},
    "C-B--C-4.9": {"x": 0, "y": fi(55, 8)},
    # "C-X--C-5.8": {"x": 0, "y": l_CX__CY + l_CY__CZ},
    "C-X--C-6.5": {"x": 0, "y": l_CX__CY + l_CY__CZ},
    "C-W.6--C-7.3": {"x": 0, "y": l_CX__CY + l_CY__CZ + l_CW6__CX},
    "C-C--C-2.5": {"x": 0, "y": fi(69, 11)},
    "C-C--C-3.6": {"x": 0, "y": fi(56, 3)},
    "C-C--C-5.2": {"x": 0, "y": fi(47, 5)},
    # "C-Y--C-5.8": {"x": 0, "y": l_CY__CZ},
    "C-Y--C-6.5": {"x": 0, "y": l_CY__CZ},
    "C-Y--C-7.2": {"x": 0, "y": l_CY__CZ},
}

In [74]:
# Calc centers of rigidity
rigid["x"] = rigid.index.get_level_values("Wall").map({key: value["x"] for key, value in wall_rigidity_loc.items()})
rigid["y"] = rigid.index.get_level_values("Wall").map({key: value["y"] for key, value in wall_rigidity_loc.items()})

rigid["kx"] = rigid["relative wall stiffness"] * rigid["x"]
rigid["CRx"] = rigid.groupby(["Level", "Direction"])["kx"].transform("sum")  # all relative wall rigidities sum to 1
rigid["ky"] = rigid["relative wall stiffness"] * rigid["y"]
rigid["CRy"] = rigid.groupby(["Level", "Direction"])["ky"].transform("sum")  # all relative wall rigidities sum to 1

# x eccentricity. Use the "+" eccentricity for walls on the + side of the CR. "-" for - side
CMx = CM_x
CMxa_plus = CM_x__a__plus
CMxa_minus = CM_x__a__minus
rigid["ex"] = np.where(rigid["x"] <= rigid["CRx"], rigid["CRx"] - CMxa_minus, CMxa_plus - rigid["CRx"])
rigid["ex"] = np.where(rigid["ex"] <= 0, abs(rigid["ex"]), rigid["ex"])  # assign 0s to negatives

# y eccentricity. Use the "+" eccentricity for walls on the + side of the CR. "-" for - side
CMy = CM_y
CMya_plus = CM_y__a__plus
CMya_minus = CM_y__a__minus
rigid["ey"] = np.where(rigid["y"] <= rigid["CRy"], rigid["CRy"] - CMya_minus, CMya_plus - rigid["CRy"])
rigid["ey"] = np.where(rigid["ey"] <= 0, abs(rigid["ey"]), rigid["ey"])  # assign 0s to negatives

# distance from CR
rigid["dx"] = abs(rigid["x"] - rigid["CRx"])
rigid["dy"] = abs(rigid["y"] - rigid["CRy"])

# Torque
rigid["torque x"] = rigid["total level cumulative shear"] * rigid["ex"]
rigid["torque y"] = rigid["total level cumulative shear"] * rigid["ey"]

# Polar moment of inertia
rigid["Jx wall"] = rigid["relative wall stiffness"] * rigid["dx"] ** 2
rigid["Jy wall"] = rigid["relative wall stiffness"] * rigid["dy"] ** 2
rigid["J"] = (rigid["Jx wall"] + rigid["Jy wall"]).groupby(["Level"]).transform("sum")

rigid["torsional shear force x"] = rigid["torque x"] * rigid["relative wall stiffness"] * rigid["dx"] / rigid["J"]
rigid["torsional shear force y"] = rigid["torque y"] * rigid["relative wall stiffness"] * rigid["dy"] / rigid["J"]
rigid["torsional shear force"] = np.where(
    rigid["torsional shear force x"] == 0, rigid["torsional shear force y"], rigid["torsional shear force x"]
)
rigid["shear force demand"] = rigid["torsional shear force"] + rigid["direct shear force"]

The centers of rigidity for each floor are given in \autoref{tab:center_of_rigidity_roof} through \autoref{tab:center_of_rigidity_level2}.

In [ ]:
column_name_map = {
    "x": "$x$ [ft]",
    "y": "$y$ [ft]",
    "CRx": "CR$_x$ [ft]",
    "dx": "$d_x$ [ft]",
    "ex": "e$_x$ [ft]",
    "CRy": "CR$_y$ [ft]",
    "dy": "$d_y$ [ft]",
    "ey": "e$_y$ [ft]",
}
display_table(
    dataframe=rigid,
    column_names_filter_and_map=column_name_map,
    levels="Level 5",
    position_float="centering",
    formatter_functions=[lambda value: sig_figs(value, sig_figs=4)],
    caption="Center of Rigidity - Level 5",
    label="tab:center_of_rigidity_level5",
)
display_table(
    dataframe=rigid,
    column_names_filter_and_map=column_name_map,
    levels="Level 4",
    position_float="centering",
    formatter_functions=[lambda value: sig_figs(value, sig_figs=4)],
    caption="Center of Rigidity - Level 4",
    label="tab:center_of_rigidity_level4",
)
display_table(
    dataframe=rigid,
    column_names_filter_and_map=column_name_map,
    levels="Level 3",
    position_float="centering",
    formatter_functions=[lambda value: sig_figs(value, sig_figs=4)],
    caption="Center of Rigidity - Level 3",
    label="tab:center_of_rigidity_level3",
)
display_table(
    dataframe=rigid,
    column_names_filter_and_map=column_name_map,
    levels="Level 2",
    position_float="centering",
    formatter_functions=[lambda value: sig_figs(value, sig_figs=4)],
    caption="Center of Rigidity - Level 2",
    label="tab:center_of_rigidity_level2",
)
display_table(
    dataframe=rigid,
    column_names_filter_and_map=column_name_map,
    levels="Level 1",
    position_float="centering",
    formatter_functions=[lambda value: sig_figs(value, sig_figs=4)],
    caption="Center of Rigidity - Level 1",
    label="tab:center_of_rigidity_level1",
)

,$x$ [ft],$y$ [ft],CR$_x$ [ft],$d_x$ [ft],e$_x$ [ft],CR$_y$ [ft],$d_y$ [ft],e$_y$ [ft]
Wall,,,,,,,,
C-2.1,182.666667,0.000000,105.739727,76.926940,7.335273,0.000000,0.000000,37.475000
C-2.9,160.000000,0.000000,105.739727,54.260273,7.335273,0.000000,0.000000,37.475000
C-3,145.083333,0.000000,105.739727,39.343607,7.335273,0.000000,0.000000,37.475000
C-3.3,137.083333,0.000000,105.739727,31.343607,7.335273,0.000000,0.000000,37.475000
C-4,118.750000,0.000000,105.739727,13.010273,7.335273,0.000000,0.000000,37.475000
C-4.2,120.333333,0.000000,105.739727,14.593607,7.335273,0.000000,0.000000,37.475000
C-5,102.166667,0.000000,105.739727,3.573060,14.814727,0.000000,0.000000,37.475000
C-5.5,104.916667,0.000000,105.739727,0.823060,14.814727,0.000000,0.000000,37.475000
C-5.9,84.291667,0.000000,105.739727,21.448060,14.814727,0.000000,0.000000,37.475000


,$x$ [ft],$y$ [ft],CR$_x$ [ft],$d_x$ [ft],e$_x$ [ft],CR$_y$ [ft],$d_y$ [ft],e$_y$ [ft]
Wall,,,,,,,,
C-2.1,182.666667,0.000000,105.381438,77.285228,7.693562,0.000000,0.000000,37.475000
C-2.9,160.000000,0.000000,105.381438,54.618562,7.693562,0.000000,0.000000,37.475000
C-3,145.083333,0.000000,105.381438,39.701895,7.693562,0.000000,0.000000,37.475000
C-3.3,137.083333,0.000000,105.381438,31.701895,7.693562,0.000000,0.000000,37.475000
C-4,118.750000,0.000000,105.381438,13.368562,7.693562,0.000000,0.000000,37.475000
C-4.2,120.333333,0.000000,105.381438,14.951895,7.693562,0.000000,0.000000,37.475000
C-5,102.166667,0.000000,105.381438,3.214772,14.456438,0.000000,0.000000,37.475000
C-5.5,104.916667,0.000000,105.381438,0.464772,14.456438,0.000000,0.000000,37.475000
C-5.9,84.291667,0.000000,105.381438,21.089772,14.456438,0.000000,0.000000,37.475000


,$x$ [ft],$y$ [ft],CR$_x$ [ft],$d_x$ [ft],e$_x$ [ft],CR$_y$ [ft],$d_y$ [ft],e$_y$ [ft]
Wall,,,,,,,,
C-2.1,182.666667,0.000000,106.400171,76.266496,6.674829,0.000000,0.000000,37.475000
C-2.9,160.000000,0.000000,106.400171,53.599829,6.674829,0.000000,0.000000,37.475000
C-3,145.083333,0.000000,106.400171,38.683162,6.674829,0.000000,0.000000,37.475000
C-3.3,137.083333,0.000000,106.400171,30.683162,6.674829,0.000000,0.000000,37.475000
C-4,118.750000,0.000000,106.400171,12.349829,6.674829,0.000000,0.000000,37.475000
C-4.2,120.333333,0.000000,106.400171,13.933162,6.674829,0.000000,0.000000,37.475000
C-5,102.166667,0.000000,106.400171,4.233504,15.475171,0.000000,0.000000,37.475000
C-5.5,104.916667,0.000000,106.400171,1.483504,15.475171,0.000000,0.000000,37.475000
C-5.9,84.291667,0.000000,106.400171,22.108504,15.475171,0.000000,0.000000,37.475000


,$x$ [ft],$y$ [ft],CR$_x$ [ft],$d_x$ [ft],e$_x$ [ft],CR$_y$ [ft],$d_y$ [ft],e$_y$ [ft]
Wall,,,,,,,,
C-2.1,182.666667,0.000000,105.023237,77.643430,8.051763,0.000000,0.000000,37.475000
C-2.9,160.000000,0.000000,105.023237,54.976763,8.051763,0.000000,0.000000,37.475000
C-3,145.083333,0.000000,105.023237,40.060097,8.051763,0.000000,0.000000,37.475000
C-3.3,137.083333,0.000000,105.023237,32.060097,8.051763,0.000000,0.000000,37.475000
C-4,118.750000,0.000000,105.023237,13.726763,8.051763,0.000000,0.000000,37.475000
C-4.2,120.333333,0.000000,105.023237,15.310097,8.051763,0.000000,0.000000,37.475000
C-5,102.166667,0.000000,105.023237,2.856570,14.098237,0.000000,0.000000,37.475000
C-5.5,104.916667,0.000000,105.023237,0.106570,14.098237,0.000000,0.000000,37.475000
C-5.9,84.291667,0.000000,105.023237,20.731570,14.098237,0.000000,0.000000,37.475000


,$x$ [ft],$y$ [ft],CR$_x$ [ft],$d_x$ [ft],e$_x$ [ft],CR$_y$ [ft],$d_y$ [ft],e$_y$ [ft]
Wall,,,,,,,,
C-2.1,182.666667,0.000000,103.741289,78.925377,9.333711,0.000000,0.000000,37.475000
C-2.9,160.000000,0.000000,103.741289,56.258711,9.333711,0.000000,0.000000,37.475000
C-3,145.083333,0.000000,103.741289,41.342044,9.333711,0.000000,0.000000,37.475000
C-3.3,137.083333,0.000000,103.741289,33.342044,9.333711,0.000000,0.000000,37.475000
C-4,118.750000,0.000000,103.741289,15.008711,9.333711,0.000000,0.000000,37.475000
C-4.2,120.333333,0.000000,103.741289,16.592044,9.333711,0.000000,0.000000,37.475000
C-5,102.166667,0.000000,103.741289,1.574623,12.816289,0.000000,0.000000,37.475000
C-5.5,104.916667,0.000000,103.741289,1.175377,9.333711,0.000000,0.000000,37.475000
C-5.9,84.291667,0.000000,103.741289,19.449623,12.816289,0.000000,0.000000,37.475000


And now, we can get the forces in the shear walls using the rigid diaphragm assumption in \autoref{tab:rigid_forces_roof} through \autoref{tab:rigid_forces_level2}. These forces can be compared to the flexible shear wall forces obtained earlier, $F_{flex}$.

In [76]:
rigid["flexible shear force demand"] = flexible["cumulative shear demand"]

In [ ]:
column_name_map = {
    "torque x": "$T_x$ [lbf-ft]",
    "torque y": "$T_y$ [lbf-ft]",
    "dx": "$d_x$ [ft]",
    "dy": "$d_y$ [ft]",
    "relative wall stiffness": "$R$ [-]",
    "torsional shear force": "$F_t$ [lbf]",
    "direct shear force": "$F_v$ [lbf]",
    "shear force demand": "$F_{tot}$ [lbf]",
    "flexible shear force demand": "$F_{flex}$ [lbf]",
}
display_table(
    dataframe=rigid,
    column_names_filter_and_map=column_name_map,
    levels="Level 5",
    position_float="centering",
    formatter_functions=[lambda value: sig_figs(value, sig_figs=4)],
    caption="Shear wall forces with the rigid diaphragm assumption - Level 5",
    label="tab:rigid_forces_level5",
)
display_table(
    dataframe=rigid,
    column_names_filter_and_map=column_name_map,
    levels="Level 4",
    position_float="centering",
    formatter_functions=[lambda value: sig_figs(value, sig_figs=4)],
    caption="Shear wall forces with the rigid diaphragm assumption - Level 4",
    label="tab:rigid_forces_level4",
)
display_table(
    dataframe=rigid,
    column_names_filter_and_map=column_name_map,
    levels="Level 3",
    position_float="centering",
    formatter_functions=[lambda value: sig_figs(value, sig_figs=4)],
    caption="Shear wall forces with the rigid diaphragm assumption - Level 3",
    label="tab:rigid_forces_level3",
)
display_table(
    dataframe=rigid,
    column_names_filter_and_map=column_name_map,
    levels="Level 2",
    position_float="centering",
    formatter_functions=[lambda value: sig_figs(value, sig_figs=4)],
    caption="Shear wall forces with the rigid diaphragm assumption - Level 2",
    label="tab:rigid_forces_level2",
)
display_table(
    dataframe=rigid,
    column_names_filter_and_map=column_name_map,
    levels="Level 1",
    position_float="centering",
    formatter_functions=[lambda value: sig_figs(value, sig_figs=4)],
    caption="Shear wall forces with the rigid diaphragm assumption - Level 1",
    label="tab:rigid_forces_level1",
)

,$T_x$ [lbf-ft],$T_y$ [lbf-ft],$d_x$ [ft],$d_y$ [ft],$R$ [-],$F_t$ [lbf],$F_v$ [lbf],$F_{tot}$ [lbf],$F_{flex}$ [lbf]
Wall,,,,,,,,,
C-2.1,4.590989e+05,2.345479e+06,76.926940,0.000000,0.109343,1617.148219,6843.566660,8460.714879,8090.047854
C-2.9,4.590989e+05,2.345479e+06,54.260273,0.000000,0.089960,938.446309,5630.391395,6568.837704,4686.795594
C-3,4.590989e+05,2.345479e+06,39.343607,0.000000,0.047010,355.583293,2942.241452,3297.824745,5693.000297
C-3.3,4.590989e+05,2.345479e+06,31.343607,0.000000,0.038590,232.544982,2415.289182,2647.834164,2098.466389
C-4,4.590989e+05,2.345479e+06,13.010273,0.000000,0.070338,175.937282,4402.329960,4578.267242,2820.021078
C-4.2,4.590989e+05,2.345479e+06,14.593607,0.000000,0.089960,252.400430,5630.391395,5882.791825,4686.795594
C-5,9.272218e+05,2.345479e+06,3.573060,0.000000,0.079189,109.865126,4956.251630,5066.116757,3543.156607
C-5.5,9.272218e+05,2.345479e+06,0.823060,0.000000,0.076833,24.554750,4808.812312,4833.367062,3334.782125
C-5.9,9.272218e+05,2.345479e+06,21.448060,0.000000,0.079091,658.672082,4950.112110,5608.784192,3534.223243


,$T_x$ [lbf-ft],$T_y$ [lbf-ft],$d_x$ [ft],$d_y$ [ft],$R$ [-],$F_t$ [lbf],$F_v$ [lbf],$F_{tot}$ [lbf],$F_{flex}$ [lbf]
Wall,,,,,,,,,
C-2.1,1.103018e+06,5.372751e+06,77.285228,0.000000,0.107919,3832.804851,15472.229058,19305.033909,18531.738943
C-2.9,1.103018e+06,5.372751e+06,54.618562,0.000000,0.082874,2080.081459,11881.540610,13961.622069,10735.965224
C-3,1.103018e+06,5.372751e+06,39.701895,0.000000,0.059258,1081.132835,8495.725930,9576.858765,13040.861712
C-3.3,1.103018e+06,5.372751e+06,31.701895,0.000000,0.041529,605.008888,5954.005125,6559.014013,4806.922282
C-4,1.103018e+06,5.372751e+06,13.368562,0.000000,0.071462,439.015710,10245.379491,10684.395201,6459.775685
C-4.2,1.103018e+06,5.372751e+06,14.951895,0.000000,0.082874,569.424729,11881.540610,12450.965339,10735.965224
C-5,2.072604e+06,5.372751e+06,3.214772,0.000000,0.076887,213.431555,11023.190575,11236.622130,8116.250293
C-5.5,2.072604e+06,5.372751e+06,0.464772,0.000000,0.075490,30.296143,10822.972919,10853.269063,7638.930311
C-5.9,2.072604e+06,5.372751e+06,21.089772,0.000000,0.076829,1399.121734,11014.947634,12414.069368,8095.786785


,$T_x$ [lbf-ft],$T_y$ [lbf-ft],$d_x$ [ft],$d_y$ [ft],$R$ [-],$F_t$ [lbf],$F_v$ [lbf],$F_{tot}$ [lbf],$F_{flex}$ [lbf]
Wall,,,,,,,,,
C-2.1,1.360216e+06,7.636764e+06,76.266496,0.000000,0.125693,5140.336325,25614.072036,30754.408361,26340.790649
C-2.9,1.360216e+06,7.636764e+06,53.599829,0.000000,0.089402,2569.540979,18218.517221,20788.058200,15259.971730
C-3,1.360216e+06,7.636764e+06,38.683162,0.000000,0.044521,923.487025,9072.557126,9996.044151,18536.123853
C-3.3,1.360216e+06,7.636764e+06,30.683162,0.000000,0.040721,669.984563,8298.231809,8968.216371,6832.501467
C-4,1.360216e+06,7.636764e+06,12.349829,0.000000,0.068534,453.853775,13966.124060,14419.977835,9181.847397
C-4.2,1.360216e+06,7.636764e+06,13.933162,0.000000,0.089402,667.946745,18218.517221,18886.463966,15259.971730
C-5,3.153575e+06,7.636764e+06,4.233504,0.000000,0.072547,381.822813,14783.845143,15165.667956,11536.340463
C-5.5,3.153575e+06,7.636764e+06,1.483504,0.000000,0.071527,131.916259,14575.887905,14707.804163,10857.883587
C-5.9,3.153575e+06,7.636764e+06,22.108504,0.000000,0.072505,1992.831766,14775.317681,16768.149447,11507.253880


,$T_x$ [lbf-ft],$T_y$ [lbf-ft],$d_x$ [ft],$d_y$ [ft],$R$ [-],$F_t$ [lbf],$F_v$ [lbf],$F_{tot}$ [lbf],$F_{flex}$ [lbf]
Wall,,,,,,,,,
C-2.1,1.979204e+06,9.211729e+06,77.643430,0.000000,0.117834,7270.908697,28964.868589,36235.777286,31773.174444
C-2.9,1.979204e+06,9.211729e+06,54.976763,0.000000,0.084399,3687.461548,20746.073689,24433.535237,18407.106691
C-3,1.979204e+06,9.211729e+06,40.060097,0.000000,0.045600,1451.736382,11208.909204,12660.645586,22358.914907
C-3.3,1.979204e+06,9.211729e+06,32.060097,0.000000,0.039261,1000.324771,9650.810291,10651.135062,8241.600030
C-4,1.979204e+06,9.211729e+06,13.726763,0.000000,0.065375,713.172677,16069.935394,16783.108072,11075.462500
C-4.2,1.979204e+06,9.211729e+06,15.310097,0.000000,0.084399,1026.895541,20746.073689,21772.969230,18407.106691
C-5,3.465487e+06,9.211729e+06,2.856570,0.000000,0.079692,316.769093,19588.976816,19905.745909,13915.533625
C-5.5,3.465487e+06,9.211729e+06,0.106570,0.000000,0.078569,11.651277,19313.140884,19324.792160,13097.155431
C-5.9,3.465487e+06,9.211729e+06,20.731570,0.000000,0.079646,2297.625797,19577.665770,21875.291566,13880.448380


,$T_x$ [lbf-ft],$T_y$ [lbf-ft],$d_x$ [ft],$d_y$ [ft],$R$ [-],$F_t$ [lbf],$F_v$ [lbf],$F_{tot}$ [lbf],$F_{flex}$ [lbf]
Wall,,,,,,,,,
C-2.1,2.519150e+06,1.011443e+07,78.925377,0.000000,0.089646,7637.647469,24195.185657,31832.833125,34886.769656
C-2.9,2.519150e+06,1.011443e+07,56.258711,0.000000,0.090949,5523.325384,24546.912308,30070.237692,20210.901252
C-3,2.519150e+06,1.011443e+07,41.342044,0.000000,0.050855,2269.539134,13725.612963,15995.152098,24549.964797
C-3.3,2.519150e+06,1.011443e+07,33.342044,0.000000,0.044514,1602.142304,12014.200774,13616.343077,9049.231210
C-4,2.519150e+06,1.011443e+07,15.008711,0.000000,0.065109,1054.861496,17572.682967,18627.544464,12160.796516
C-4.2,2.519150e+06,1.011443e+07,16.592044,0.000000,0.090949,1628.961211,24546.912308,26175.873519,20210.901252
C-5,3.459091e+06,1.011443e+07,1.574623,0.000000,0.079101,184.619797,21349.110667,21533.730464,15279.178889
C-5.5,2.519150e+06,1.011443e+07,1.175377,0.000000,0.078080,99.066838,21073.517777,21172.584615,14380.604162
C-5.9,3.459091e+06,1.011443e+07,19.449623,0.000000,0.079059,2279.204210,21337.822458,23617.026668,15240.655485


### Final Shear Wall Nailing Selection

With the envelope procedure complete, we can now select the shear wall sheathing and nailing specifications in \autoref{tab:shear_walls_roof} through \autoref{tab:shear_walls_level2}.

In [ ]:
shear_walls["wall length"] = rigid["wall length"]
shear_walls["rigid shear force demand"] = rigid["shear force demand"]
shear_walls["flexible shear force demand"] = rigid["flexible shear force demand"]
shear_walls["shear force demand"] = np.where(
    shear_walls["flexible shear force demand"] >= shear_walls["rigid shear force demand"],
    shear_walls["flexible shear force demand"],
    shear_walls["rigid shear force demand"],
)
shear_walls["unit shear demand"] = shear_walls["shear force demand"] / shear_walls["wall length"]
shear_walls["adjusted unit shear demand"] = (
    shear_walls["shear force demand"] / shear_walls["wall length"] * LOAD_FACTOR_SEISMIC_ASD
)
shear_walls["sheathed sides"] = 1

# using sheathing from the NS calc in memory

shear_walls[["adjusted unit shear capacity", "nail spacing", "shear stiffness", "shear dcr"]] = (
    get_sheathing_properties(
        shear_walls[["adjusted unit shear demand", "sheathed sides"]],
        sheathing=sheathing,
    )
)

# Assign 2-sided sheathing to walls requiring it, then recalc capacity for just those walls (more efficient than passing the full wall dataframe)
if (shear_walls["shear dcr"] > shear_dcr_check_value).any():
    shear_walls.loc[shear_walls["shear dcr"] > shear_dcr_check_value, "sheathed sides"] = 2
    shear_walls.loc[
        shear_walls["shear dcr"] > shear_dcr_check_value,
        ["adjusted unit shear capacity", "nail spacing", "shear stiffness", "shear dcr"],
    ] = get_sheathing_properties(
        shear_walls.loc[
            shear_walls["shear dcr"] > shear_dcr_check_value,
            ["adjusted unit shear demand", "sheathed sides"],
        ],
        sheathing=sheathing,
    )

column_name_map = {
    "shear force demand": "$F_{x,cum}$ [lbf]",
    "wall length": "$l_{wall}$ [ft]",
    "unit shear demand": "$v$ [plf]",
    "adjusted unit shear demand": "$0.7 v$ [plf]",
    "adjusted unit shear capacity": "$v_{cap}$ [plf]",
    "sheathed sides": "sides",
    "nail spacing": "$s_{nail}$ [in]",
}

display_table(
    dataframe=shear_walls,
    column_names_filter_and_map=column_name_map,
    levels="Level 5",
    position_float="centering",
    caption="Final shear wall design values - Level 5",
    label="tab:shear_walls_level5",
)
display_table(
    dataframe=shear_walls,
    column_names_filter_and_map=column_name_map,
    levels="Level 4",
    position_float="centering",
    caption="Final shear wall design values - Level 4",
    label="tab:shear_walls_level4",
)
display_table(
    dataframe=shear_walls,
    column_names_filter_and_map=column_name_map,
    levels="Level 3",
    position_float="centering",
    caption="Final shear wall design values - Level 3",
    label="tab:shear_walls_level3",
)
display_table(
    dataframe=shear_walls,
    column_names_filter_and_map=column_name_map,
    levels="Level 2",
    position_float="centering",
    caption="Final shear wall design values - Level 2",
    label="tab:shear_walls_level2",
)
display_table(
    dataframe=shear_walls,
    column_names_filter_and_map=column_name_map,
    levels="Level 1",
    position_float="centering",
    caption="Final shear wall design values - Level 1",
    label="tab:shear_walls_level1",
)

,"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [plf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,
C-2.1,8460.714879,29.000000,291.748789,204.224152,310.714286,1,6.0
C-2.9,6568.837704,29.000000,226.511645,158.558151,310.714286,1,6.0
C-3,5693.000297,14.250000,399.508793,279.656155,310.714286,1,6.0
C-3.3,2647.834164,19.000000,139.359693,97.551785,310.714286,1,6.0
C-4,4578.267242,29.000000,157.871284,110.509899,310.714286,1,6.0
C-4.2,5882.791825,29.000000,202.854891,141.998423,310.714286,1,6.0
C-5,5066.116757,29.000000,174.693681,122.285577,310.714286,1,6.0
C-5.5,4833.367062,29.000000,166.667830,116.667481,310.714286,1,6.0
C-5.9,5608.784192,29.000000,193.406351,135.384446,310.714286,1,6.0


,"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [plf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,
C-2.1,19305.033909,29.000000,665.690824,465.983577,600.000000,1,3.0
C-2.9,13961.622069,29.000000,481.435244,337.004671,460.714286,1,4.0
C-3,13040.861712,14.250000,915.148190,640.603733,769.642857,1,2.0
C-3.3,6559.014013,19.000000,345.211264,241.647885,310.714286,1,6.0
C-4,10684.395201,29.000000,368.427421,257.899195,310.714286,1,6.0
C-4.2,12450.965339,29.000000,429.343632,300.540543,310.714286,1,6.0
C-5,11236.622130,29.000000,387.469729,271.228810,310.714286,1,6.0
C-5.5,10853.269063,29.000000,374.250657,261.975460,310.714286,1,6.0
C-5.9,12414.069368,29.000000,428.071358,299.649950,310.714286,1,6.0


,"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [plf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,
C-2.1,30754.408361,29.000000,1060.496840,742.347788,769.642857,1,2.0
C-2.9,20788.058200,29.000000,716.829593,501.780715,600.000000,1,3.0
C-3,18536.123853,14.250000,1300.780621,910.546435,921.428571,2,4.0
C-3.3,8968.216371,19.000000,472.011388,330.407972,460.714286,1,4.0
C-4,14419.977835,29.000000,497.240615,348.068431,460.714286,1,4.0
C-4.2,18886.463966,29.000000,651.257378,455.880165,460.714286,1,4.0
C-5,15165.667956,29.000000,522.954067,366.067847,460.714286,1,4.0
C-5.5,14707.804163,29.000000,507.165661,355.015963,460.714286,1,4.0
C-5.9,16768.149447,29.000000,578.212050,404.748435,460.714286,1,4.0


,"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [plf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,
C-2.1,36235.777286,29.000000,1249.509562,874.656693,921.428571,2,4.0
C-2.9,24433.535237,29.000000,842.535698,589.774988,600.000000,1,3.0
C-3,22358.914907,14.250000,1569.046660,1098.332662,1200.000000,2,3.0
C-3.3,10651.135062,19.000000,560.586056,392.410239,460.714286,1,4.0
C-4,16783.108072,29.000000,578.727865,405.109505,460.714286,1,4.0
C-4.2,21772.969230,29.000000,750.792042,525.554430,600.000000,1,3.0
C-5,19905.745909,29.000000,686.405031,480.483522,600.000000,1,3.0
C-5.5,19324.792160,29.000000,666.372143,466.460500,600.000000,1,3.0
C-5.9,21875.291566,29.000000,754.320399,528.024279,600.000000,1,3.0


,"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [plf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,
C-2.1,34886.769656,29.000000,1202.992057,842.094440,921.428571,2,4.0
C-2.9,30070.237692,29.000000,1036.904748,725.833324,769.642857,1,2.0
C-3,24549.964797,14.250000,1722.804547,1205.963183,1539.285714,2,2.0
C-3.3,13616.343077,19.000000,716.649636,501.654745,600.000000,1,3.0
C-4,18627.544464,29.000000,642.329119,449.630384,460.714286,1,4.0
C-4.2,26175.873519,29.000000,902.616328,631.831430,769.642857,1,2.0
C-5,21533.730464,29.000000,742.542430,519.779701,600.000000,1,3.0
C-5.5,21172.584615,29.000000,730.089125,511.062387,600.000000,1,3.0
C-5.9,23617.026668,29.000000,814.380230,570.066161,600.000000,1,3.0


### Shear Wall Axial Forces

#### Tension Forces

The shear forces get converted to overturning forces in the end posts of the shear walls. This overturning moment is counteracted by the dead load tributary to each shear wall according to ASCE 7-16 Sec. 2.4.5. The applicable equation is:

$$
0.6 D - 0.7 E_v + 0.7 E_h \quad\text{(ASCE 7-16 Eq. 2.4.5-10)}
$$

There are no vertical seismic effects, $E_v$, so we will only consider the horizontal seismic effects, $E_h$, and dead loads, $D$.

The overturning moment is:

$$
M_{OT} = V h_{wall}
$$

Using the area dead loads from earlier, along with the tributary areas for each shear wall, we can get the dead load that is applied to each wall. This resists the overturning moment at the centroid of the wall.

$$
M_R = \frac{D l_{wall}}{2}
$$

We will assume that the holddowns are 6" inset from the end posts. The tension caused by horizontal load effects counteracted by dead loads is:

$$
T_{seismic} = \frac{M_{OT} - M_R}{l_{wall} - 0.5}\quad\text{(units in feet)}
$$

In [79]:
shear_walls["line"] = flexible["line"]
shear_walls["wall height"] = rigid["wall height"]
shear_walls["tributary area"] = flexible["tributary area"]
shear_walls["floor OTM"] = shear_walls["shear force demand"] * shear_walls["wall height"]
shear_walls["OTM"] = shear_walls.groupby(level="Wall")["floor OTM"].cumsum()
shear_walls["dead load"] = 0.6 * float(U_floor * 1000) * shear_walls["tributary area"]

# Reduce resisting moment dead load in some walls that definitely are not correct
shear_walls.loc[shear_walls.index.get_level_values("Wall") == "C-2.1", "dead load"] *= 1 / 4
shear_walls.loc[shear_walls.index.get_level_values("Wall") == "C-2.9", "dead load"] *= 1 / 2
shear_walls.loc[shear_walls.index.get_level_values("Wall") == "C-3", "dead load"] *= 1 / 2
shear_walls.loc[shear_walls.index.get_level_values("Wall") == "C-4.2", "dead load"] *= 1 / 4
shear_walls.loc[shear_walls.index.get_level_values("Wall") == "C-6", "dead load"] *= 1 / 4
shear_walls.loc[shear_walls.index.get_level_values("Wall") == "C-7", "dead load"] *= 1 / 4
shear_walls.loc[shear_walls.index.get_level_values("Wall") == "C-7.3", "dead load"] *= 1 / 8
shear_walls.loc[shear_walls.index.get_level_values("Wall") == "C-8", "dead load"] *= 1 / 4
shear_walls.loc[shear_walls.index.get_level_values("Wall") == "C-8.1", "dead load"] *= 1 / 4
shear_walls.loc[shear_walls.index.get_level_values("Wall") == "C-8.3", "dead load"] *= 1 / 4
shear_walls.loc[shear_walls["line"] == "C-B", "dead load"] *= 0
shear_walls.loc[shear_walls["line"] == "C-C", "dead load"] *= 0
shear_walls.loc[shear_walls["line"] == "C-X", "dead load"] *= 0
shear_walls.loc[shear_walls["line"] == "C-Y", "dead load"] *= 0

shear_walls["RM"] = shear_walls["dead load"] * shear_walls["wall length"] / 2

shear_walls["total moment"] = shear_walls["OTM"] - shear_walls["RM"]
shear_walls["total moment"] = np.where(
    shear_walls["RM"] <= shear_walls["OTM"], shear_walls["OTM"] - shear_walls["RM"], 0
)
wall_arm_shortening_length = 0.5  # feet due to holddown positioning
shear_walls["tension force"] = shear_walls["total moment"] / (shear_walls["wall length"] - wall_arm_shortening_length)
num_bins = 5
shear_walls["binned tension force"] = (
    shear_walls["tension force"]
    .groupby(level="Level")
    .transform(lambda s: pd.qcut(s, q=num_bins, duplicates="drop").map(lambda x: x.right if pd.notna(x) else 0))
)

#### Compression Forces

We will calculate the compression forces in a very similar way. However, we will ignore the resisting moment.

The live load, $L$, is $40$ psf, while the snow load, $S$, is $25$ psf and applies only to corridor walls as that is how the roof trusses are supported.

$$
C_{seismic} = \frac{M_{OT}}{l_{wall} - 0.5}\quad\text{(units in feet)}
$$

In [80]:
shear_walls["compression force"] = shear_walls["OTM"] / (shear_walls["wall length"] - wall_arm_shortening_length)
shear_walls["binned compression force"] = (
    shear_walls["compression force"]
    .groupby(level="Level")
    .transform(lambda s: pd.qcut(s, q=num_bins).map(lambda x: x.right if pd.notna(x) else 0))
    # .transform(lambda s: pd.qcut(s, q=num_bins, duplicates="drop").map(lambda x: x.right if pd.notna(x) else pd.NA))
)

#### Holddown Forces Table

With the tension and compression forces calculated, we can construct a table of the forces for each level. We will default to 5 bins for each level to give us options A through E.

However, there is only one table in the plans for all the holddown forces, so these will all get compared together as an aggregate of all buildings and simplified further.

In [81]:
NUM_OPTIONS = 5
LABELS = list("ABCDE")

schedule_rows = []


def assign_schedule(group):
    group = group.copy()

    level = group.index.get_level_values("Level")[0]

    n = len(group)
    group["_bin"] = np.floor(np.arange(n) * min(NUM_OPTIONS, n) / n)

    schedule = (
        group.groupby("_bin")
        .agg(
            tension=("tension force", "max"),
            compression=("compression force", "max"),
        )
        .reset_index()
    )

    # convert to kips
    schedule["tension"] /= 1000
    schedule["compression"] /= 1000

    # sort ONLY schedule
    schedule["governing"] = schedule[["tension", "compression"]].max(axis=1)
    schedule = schedule.sort_values("governing").reset_index(drop=True)

    # assign A–E
    schedule["option"] = LABELS[: len(schedule)]

    # IMPORTANT: keep level in schedule
    schedule["Level"] = level

    # map bins → options
    bin_to_option = dict(zip(schedule["_bin"], schedule["option"]))
    group["option"] = group["_bin"].map(bin_to_option)

    schedule_map = schedule.set_index("option")[["tension", "compression"]]

    group["binned tension force"] = group["option"].map(schedule_map["tension"])
    group["binned compression force"] = group["option"].map(schedule_map["compression"])

    schedule_rows.append(schedule[["Level", "option", "tension", "compression"]])

    return group.drop(columns=["_bin"])


shear_walls = shear_walls.groupby(level="Level", group_keys=False).apply(assign_schedule)

schedule_df = (
    pd.concat(schedule_rows, ignore_index=True)
    .rename(
        columns={
            "tension": "binned tension force",
            "compression": "binned compression force",
        }
    )
    .set_index(["Level", "option"])
    .sort_index()
)

In [82]:
column_name_map = {
    "binned tension force": "$T$ [kip]",
    "binned compression force": "$C$ [kip]",
}
display_table(
    schedule_df,
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Holddown Schedule",
    label="tab:holddown_schedule",
)

$T$ [kip]  $C$ [kip]
Level   option                      
Level 2 A       31.493764  34.133431
        B       63.107041  69.638399
        C       64.783474  71.712441
        D       83.234720  83.234720
        E       81.077112  95.414065
Level 3 A       19.094666  21.734333
        B       39.003439  45.534797
        C       44.479433  46.109723
        D       54.244292  54.244292
        E       48.051903  62.388856
Level 4 A       11.073045  13.712713
        B       21.929359  28.460716
        C       27.149385  29.074106
        D       33.977613  33.977613
        E       24.658094  38.995047
Level 5 A        4.513969   6.754542
        B        7.774500  14.305858
        C       12.217729  14.951135
        D       17.590878  17.590878
        E       10.947555  19.600969
Roof    A        0.119885   2.167344
        B        3.915331   4.337589
        C        0.000000   4.347382
        D        5.412902   5.412902
        E        2.559954   5.956504

## Diaphragm Design

We must first get the diaphragm inertial design forces. These are distinct from the seismic forces determined in \autoref{tab:seismic_forces_calculation_table}.

\begin{align*}
F_{px} &= \frac{\sum_{i=x}^n F_i}{\sum_{i=x}^n w_i} w_{px} \quad \text{(ASCE Eq. 12.10-1)} \\
F_{px} &\ge 0.2 S_{DS} I_e w_{px} \quad \text{(ASCE Eq. 12.10-2)} \\
F_{px} &\le 0.4 S_{DS} I_e w_{px}  \quad \text{(ASCE Eq. 12.10-3)}
\end{align*}

These forces are shown in \autoref{tab:diaphragm_forces}

In [83]:
seismic_loads["cumulative floor weight"] = seismic_loads.iloc[::-1]["floor weight"].cumsum()
seismic_loads["F_x,cum"] = seismic_loads.iloc[::-1]["F_x"].cumsum()
seismic_loads["F_px lower bound"] = 0.2 * S_DS * I_e * seismic_loads["floor weight"]
seismic_loads["F_px upper bound"] = 0.4 * S_DS * I_e * seismic_loads["floor weight"]
seismic_loads["F_px initial"] = (
    seismic_loads["F_x,cum"] / seismic_loads["cumulative floor weight"] * seismic_loads["floor weight"]
)

# Assign F_px with limits from ASCE
seismic_loads["F_px"] = np.where(
    seismic_loads["F_px initial"] <= seismic_loads["F_px lower bound"],
    seismic_loads["F_px lower bound"],
    np.where(
        seismic_loads["F_px initial"] >= seismic_loads["F_px upper bound"],
        seismic_loads["F_px upper bound"],
        seismic_loads["F_px initial"],
    ),
)

In [84]:
column_name_map = {
    "floor weight": "$w_{px}$ [kip]",
    "cumulative floor weight": r"$\sum w_x$ [kip]",
    "F_x,cum": r"$\sum F_x$ [kip]",
    "F_px initial": "$F_{px,init}$ [kip]",
    "F_px lower bound": "$F_{px,lower}$ [kip]",
    "F_px upper bound": "$F_{px,upper}$ [kip]",
    "F_px": "$F_{px}$ [kip]",
}
display_table(
    dataframe=seismic_loads.iloc[:0:-1],
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Diaphragm Forces",
    label="tab:diaphragm_forces",
)

,$w_{px}$ [kip],$\sum w_x$ [kip],$\sum F_x$ [kip],"$F_{px,init}$ [kip]","$F_{px,lower}$ [kip]","$F_{px,upper}$ [kip]",$F_{px}$ [kip]
Level,,,,,,,
Roof,348.640312,348.640312,61.317250,61.317250,47.219491,94.438981,61.317250
Level 5,554.980625,903.620937,140.458411,86.265926,75.166013,150.332027,86.265926
Level 4,541.375625,1444.996562,199.645895,74.798393,73.323366,146.646732,74.798393
Level 3,541.375625,1986.372188,240.819796,65.634209,73.323366,146.646732,73.323366
Level 2,551.630000,2538.002188,264.418803,57.470929,74.712208,149.424416,74.712208


With these diaphragm forces, we can design the diaphragm for the worst-case cantilever and simply-supported conditions.

The longest simply-supported span in the N-S direction is about 34.5 feet, while the longest cantilever span is about 32 feet. The depth of the diaphragm in the N-S direction is 67 feet. The longest simply-supported span in the E-W direction is about 5.5 feet, while the longest cantilever span is about 31 feet. The depth of the diaphragm in the E-W direction is 230 feet.

Diaphragm unit shear demands are shown in \autoref{tab:diaphragm_unit_shears}

In [ ]:
# Set up diaphragm dataframe
levels = ["Level 5", "Level 4", "Level 3", "Level 2", "Level 1"]
directions = ["N-S", "E-W"]
index = pd.MultiIndex.from_product(
    [levels, directions],
    names=["Level", "Direction"],
)
diaphragms = pd.DataFrame(index=index)
diaphragms_ns = diaphragms.xs("N-S", level="Direction").copy()
diaphragms_ew = diaphragms.xs("E-W", level="Direction").copy()

In [86]:
diaphragms_ns["F_px"] = seismic_loads["F_px"] * 1000
diaphragms_ns["area load"] = diaphragms_ns["F_px"] / float(A_level)
diaphragms_ns["L_simp"] = 35
diaphragms_ns["L_cant"] = 35
diaphragms_ns["depth"] = 67
diaphragms_ns["V_simp"] = diaphragms_ns["area load"] * diaphragms_ns["depth"] * diaphragms_ns["L_simp"] / 2
diaphragms_ns["V_cant"] = diaphragms_ns["area load"] * diaphragms_ns["depth"] * diaphragms_ns["L_cant"]
diaphragms_ns["v_simp"] = diaphragms_ns["V_simp"] / diaphragms_ns["depth"]
diaphragms_ns["v_cant"] = diaphragms_ns["V_cant"] / diaphragms_ns["depth"]

We will assume the diaphragm sheathing has the following properties:

- Unblocked
- WSP Sheathing
- 15/32" thick
- 10d common nails
- Plywood
- Nailed face is 2"
- Edge adjoining cases 2 through 6

The diaphragm capacity checks for this chosen sheathing are shown in \autoref{tab:diaphragm_capacities}

In [87]:
# Set baseline sheathing properties. We set the number of sheathed sides later
from structural_tools.seismic.timber.sheathing import SheathingApplication

diaphragm_sheathing = Sheathing(
    sheathing_material=SheathingMaterial.WSP_SHEATHING,
    minimum_nominal_panel_thickness=15 / 32,
    nail=Nail.COMMON_10D,
    panel_type=PanelType.PLY,
    minimum_nominal_width_of_nailed_face=2,
    adjoining_panel_edge_location=(2, 3, 4, 5, 6),
)

diaphragms_ns[
    ["adjusted unit shear capacity simply supported", "shear stiffness simply supported", "shear dcr simply supported"]
] = get_sheathing_properties(
    diaphragms_ns["v_simp"], sheathing=sheathing, sheathing_application=SheathingApplication.DIAPHRAGM
)

diaphragms_ns[["adjusted unit shear capacity cantilever", "shear stiffness cantilever", "shear dcr cantilever"]] = (
    get_sheathing_properties(
        diaphragms_ns["v_cant"], sheathing=sheathing, sheathing_application=SheathingApplication.DIAPHRAGM
    )
)

In [88]:
diaphragms_ew["F_px"] = seismic_loads["F_px"] * 1000
diaphragms_ew["area load"] = diaphragms_ew["F_px"] / float(A_level)
diaphragms_ew["L_simp"] = 5.5
diaphragms_ew["L_cant"] = 35
diaphragms_ew["depth"] = X_plan
diaphragms_ew["V_simp"] = diaphragms_ew["area load"] * diaphragms_ew["depth"] * diaphragms_ew["L_simp"] / 2
diaphragms_ew["V_cant"] = diaphragms_ew["area load"] * diaphragms_ew["depth"] * diaphragms_ew["L_cant"]
diaphragms_ew["v_simp"] = diaphragms_ew["V_simp"] / diaphragms_ew["depth"]
diaphragms_ew["v_cant"] = diaphragms_ew["V_cant"] / diaphragms_ew["depth"]


In [89]:
diaphragms_ew[
    ["adjusted unit shear capacity simply supported", "shear stiffness simply supported", "shear dcr simply supported"]
] = get_sheathing_properties(
    diaphragms_ew["v_simp"], sheathing=sheathing, sheathing_application=SheathingApplication.DIAPHRAGM
)

diaphragms_ew[["adjusted unit shear capacity cantilever", "shear stiffness cantilever", "shear dcr cantilever"]] = (
    get_sheathing_properties(
        diaphragms_ew["v_cant"], sheathing=sheathing, sheathing_application=SheathingApplication.DIAPHRAGM
    )
)

In [90]:
idx = pd.IndexSlice
diaphragms = diaphragms.reindex(columns=diaphragms_ns.columns)
diaphragms_ns["Direction"] = "N-S"
diaphragms_ew["Direction"] = "E-W"
diaphragms_ns = diaphragms_ns.set_index("Direction", append=True)
diaphragms_ew = diaphragms_ew.set_index("Direction", append=True)
diaphragms.update(diaphragms_ns)
diaphragms.update(diaphragms_ew)

In [91]:
column_name_map = {
    "F_px": "$F_{px}$ [lbf]",
    "area load": "$w$ [psf]",
    "V_simp": "$V_{simp}$ [lbf]",
    "V_cant": "$V_{cant}$ [lbf]",
    "v_simp": "$v_{simp}$ [plf]",
    "v_cant": "$v_{cant}$ [plf]",
}
display_table(
    dataframe=diaphragms,
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Diaphragm Unit Shear Demands",
    label="tab:diaphragm_unit_shears",
)

$F_{px}$ [lbf]  $w$ [psf]  $V_{simp}$ [lbf]  \
Level   Direction                                                
Roof    N-S          61317.249898   4.570116       5358.461318   
        E-W          61317.249898   4.570116       2783.772069   
Level 5 N-S          86265.925872   6.429599       7538.704486   
        E-W          86265.925872   6.429599       3916.429313   
Level 4 N-S          74798.393210   5.574897       6536.566747   
        E-W          74798.393210   5.574897       3395.809142   
Level 3 N-S          73323.366056   5.464960       6407.665402   
        E-W          73323.366056   5.464960       3328.843657   
Level 2 N-S          74712.208215   5.568473       6529.035115   
        E-W          74712.208215   5.568473       3391.896387   

                   $V_{cant}$ [lbf]  $v_{simp}$ [plf]  $v_{cant}$ [plf]  
Level   Direction                                                        
Roof    N-S            10716.922636         79.977035        159.954069  
        E-W            35429.826327         12.567820        159.954069  
Level 5 N-S            15077.408971        112.517977        225.035955  
        E-W            49845.463987         17.681396        225.035955  
Level 4 N-S            13073.133493         97.560698        195.121395  
        E-W            43219.389086         15.330967        195.121395  
Level 3 N-S            12815.330804         95.636797        191.273594  
        E-W            42367.101092         15.028640        191.273594  
Level 2 N-S            13058.070229         97.448285        194.896571  
        E-W            43169.590384         15.313302        194.896571

In [92]:
column_name_map = {
    "v_simp": "$v_{simp}$ [plf]",
    "adjusted unit shear capacity simply supported": "$v_{cap,simp}$ [plf]",
    "shear dcr simply supported": "DCR$_{simp}$ [-]",
    "v_cant": "$v_{cant}$ [plf]",
    "adjusted unit shear capacity cantilever": "$v_{cap,cant}$ [plf]",
    "shear dcr cantilever": "DCR$_{cant}$ [-]",
}
display_table(
    dataframe=diaphragms,
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Diaphragm Capacity Checks",
    label="tab:diaphragm_capacities",
)

$v_{simp}$ [plf]  $v_{cap,simp}$ [plf]  DCR$_{simp}$ [-]  \
Level   Direction                                                             
Roof    N-S               79.977035            189.285714          0.422520   
        E-W               12.567820            189.285714          0.066396   
Level 5 N-S              112.517977            189.285714          0.594435   
        E-W               17.681396            189.285714          0.093411   
Level 4 N-S               97.560698            189.285714          0.515415   
        E-W               15.330967            189.285714          0.080994   
Level 3 N-S               95.636797            189.285714          0.505251   
        E-W               15.028640            189.285714          0.079397   
Level 2 N-S               97.448285            189.285714          0.514821   
        E-W               15.313302            189.285714          0.080900   

                   $v_{cant}$ [plf]  $v_{cap,cant}$ [plf]  DCR$_{cant}$ [-]  
Level   Direction                                                            
Roof    N-S              159.954069            189.285714          0.845040  
        E-W              159.954069            189.285714          0.845040  
Level 5 N-S              225.035955            255.357143          0.881260  
        E-W              225.035955            255.357143          0.881260  
Level 4 N-S              195.121395            214.285714          0.910567  
        E-W              195.121395            214.285714          0.910567  
Level 3 N-S              191.273594            214.285714          0.892610  
        E-W              191.273594            214.285714          0.892610  
Level 2 N-S              194.896571            214.285714          0.909517  
        E-W              194.896571            214.285714          0.909517

In [93]:
if (
    (diaphragms[["shear dcr simply supported", "shear dcr cantilever"]] <= 1).all().all()
):  # double .all() because 2 columns
    display_text(r"$\therefore$ All shear capacities are OK")
else:
    display_text("You must check for new diaphragm nailing.")

'$\\therefore$ All shear capacities are OK'

## Story Drift Determination

The inelastic design story drift is calculated as the difference in deflections between stories at their centers of mass.

We can find the shear wall line that is closest to the center of mass, then take the cumulative force on that line and divide it by the cumulative wall stiffness on that line. This will give us an approximate elastic story drift that will then be converted to inelastic story drift with $C_d$ and $I_e$.

Note that this is a conservative estimation of inelastic story drift due to assuming the deflection of a single shear wall line, rather than the full deflection of the diaphragm as a rigid unit.

\begin{align*}
    \delta_{xe} &= F/k \\
    \delta_x &= \frac{C_d \delta_{xe}}{I_e}
\end{align*}

In [ ]:
%%render
C_d
I_e

<IPython.core.display.Latex object>

In [ ]:
shear_walls["line"] = flexible["line"]
shear_walls["wall stiffness"] = rigid["wall stiffness"]
shear_walls["cumulative shear demand"] = flexible["cumulative shear demand"]
shear_walls["x"] = rigid["x"]
shear_walls["y"] = rigid["y"]
shear_walls["delta_sw"] = rigid["delta_sw"]


def line_deflection(group, line):
    line_walls = group[group["line"] == line]

    total_stiffness = line_walls["wall stiffness"].sum()
    total_shear = line_walls["cumulative shear demand"].sum()

    return total_shear / total_stiffness


def get_cm_line_deflection(group, CM_x, CM_y, C_d, I_e):
    x_wall = group.loc[(group["x"] - CM_x).abs().idxmin()]

    y_wall = group.loc[(group["y"] - CM_y).abs().idxmin()]

    x_line = x_wall["line"]
    y_line = y_wall["line"]

    x_delta_sw = line_deflection(group, x_line)
    y_delta_sw = line_deflection(group, y_line)

    drift_factor = C_d / I_e

    return pd.Series(
        {
            "x line": x_line,
            "x delta_sw": x_delta_sw,
            "x inelastic drift": x_delta_sw * drift_factor,
            "y line": y_line,
            "y delta_sw": y_delta_sw,
            "y inelastic drift": y_delta_sw * drift_factor,
        }
    )


story_drift = shear_walls.groupby(level="Level", sort=False).apply(
    get_cm_line_deflection,
    CM_x=CM_x,
    CM_y=CM_y,
    C_d=C_d,
    I_e=I_e,
)
story_drift["allowable story drift"] = seismic_loads["floor height"] * 0.02 * 12

In [ ]:
column_name_map = {
    "x line": "N-S line",
    "x delta_sw": r"$\delta_{xe,ns}$ [in]",
    "x inelastic drift": r"$\delta_{x,ns}$ [in]",
    "y line": "E-W line",
    "y delta_sw": r"$\delta_{xe,ew}$ [in]",
    "y inelastic drift": r"$\delta_{x,ew}$ [in]",
    "allowable story drift": r"$\Delta_a$ [in]",
}
display_table(
    story_drift,
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Story Drifts",
    label="tab:story_drifts",
)

,N-S line,"$\delta_{sw,ns}$ [in]","$\delta_{x,ns}$ [in]",E-W line,"$\delta_{sw,ew}$ [in]","$\delta_{x,ew}$ [in]",$\Delta_a$ [in]
Level,,,,,,,
Level 4,A-6,0.197989,0.791958,A-B,0.374226,1.496904,2.40
Level 3,A-6,0.341671,1.366682,A-B,0.565314,2.261256,2.52
Level 2,A-6,0.373059,1.492237,A-B,0.653255,2.613021,2.52
Level 1,A-6,0.453864,1.815455,A-B,0.697459,2.789837,2.76


In [ ]:
# Save dataframes to CSV
shear_walls.to_csv("shear_walls_C.csv")